## 3 - Data Validation Generico

Aplica los estándares de preparación de datos (E) y los criterios de implementación (C5-C10) sobre el `PanelCapacidades`, y produce el `PanelModelo` y el archivo de decisiones que consumen los notebooks de modelado.

# Parámetros

Único bloque a editar por embotellador-país.

In [ ]:
pip install pyfixest

In [ ]:
BU = 'MEX'
CAPACIDADES_CONTINUAS_STR = 'Digital,Multicategory,Coolers,PedidoSugerido,GuidedMissions,Loyalty'
CAPACIDADES_BINARIAS_STR  = 'POS,DigitalServices,RTM' 

In [ ]:
CAPACIDADES_CONTINUAS = CAPACIDADES_CONTINUAS_STR.split(',')
CAPACIDADES_BINARIAS  = CAPACIDADES_BINARIAS_STR.split(',')
CAPACIDADES_MODELO    = CAPACIDADES_CONTINUAS + CAPACIDADES_BINARIAS

# Capacidades de stock: reciben retro-completado en E5
CAPACIDADES_STOCK = ['Coolers']

# Flags de fecha de adopción no confiable que trae el PanelCapacidades
FLAGS_CENSURA = {'POS': 'pos_fecha_no_confiable', 'DigitalServices': 'ds_fecha_no_confiable'}

UMBRALES = {
    'masa_pura_min_pct': 0.1,     # C8
    'vif_max':           10,      # C8
    'r2_max':            0.95,    # C8
    'cond_xtx_max':      1e10,    # C8
    'streak_min':        6,       # C9
    'pdvs_min_pct':      0.5,     # C9
    'penetracion_max':   0.90,    # C10
    'saturacion_max':    0.80,    # C10
}

N_PDVS_MUESTRA_VIF = 50_000
OUTCOME      = 'ingreso_neto_core'
OUTCOME_REAL = f'{OUTCOME}_real'

USE_MUESTRA = False
PCT_MUESTRA = 0.2

# Setup y Lectura de Datos

In [ ]:
from pyspark.sql import functions as sf
from pyspark.sql import Window
from pyspark.storagelevel import StorageLevel
import datetime
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import seaborn as sns
import scipy.stats as scistats
from scipy.stats import norm
import re
import pyfixest as pf
from itertools import combinations
import json

RUN_QA = True

spark.conf.set("fs.azure.account.key." + storageAccountName + ".blob.core.windows.net", sas)
spark.sql("set spark.sql.legacy.timeParserPolicy=LEGACY")

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "104857600")

plt.rcParams.update({
    "figure.figsize":     (14, 5),
    "figure.titlesize":   13,
    "figure.titleweight": "bold",
    "axes.titlesize":     11,
    "axes.titleweight":   "normal",
    "axes.labelsize":     10,
    "grid.linestyle":     "--",
    "grid.alpha":         0.3,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "legend.fontsize":    9,
    "font.size":          10,
})

COL = {
    "serie":   ["#2C5F8A", "#3E8E9C", "#D2772F", "#6E97B8", "#5A7D3C", "#8A8D8F"],
    "primary": "#2C5F8A",
    "accent":  "#D2772F",
    "ok":      "#2E7D52",
    "warn":    "#D98A2B",
    "alert":   "#B23B3B",
    "ref":     "#7A7A7A",
}
CMAP_SEC = "Blues"
CMAP_DIV = "vlag"

print(f"BU configurado:        {BU}")
print(f"Capacidades continuas: {CAPACIDADES_CONTINUAS}")
print(f"Capacidades binarias:  {CAPACIDADES_BINARIAS}")
print(f"Outcome:               {OUTCOME}")
print(f"Muestra:               {'Sí (' + str(int(PCT_MUESTRA*100)) + '%)' if USE_MUESTRA else 'No (panel completo)'}")

In [ ]:
input_path = f"abfss://{containerName}@{storageAccountName}.dfs.core.windows.net/CTG/{BU}/PanelCapacidades/parquet/"
panel_qa = spark.read.parquet(input_path)

# Capacidades efectivamente presentes en este BU
CAPACIDADES_CONTINUAS = [c for c in CAPACIDADES_CONTINUAS if c in panel_qa.columns]
CAPACIDADES_BINARIAS  = [c for c in CAPACIDADES_BINARIAS  if c in panel_qa.columns]
CAPACIDADES_MODELO    = CAPACIDADES_CONTINUAS + CAPACIDADES_BINARIAS
CAPS = CAPACIDADES_CONTINUAS + CAPACIDADES_BINARIAS

# Medido-desde: primer periodo con valor no nulo por capacidad
md_exprs = [sf.min(sf.when(sf.col(c).isNotNull(), sf.col("periodo"))).alias(c) for c in CAPS]
md_row = panel_qa.agg(*md_exprs).collect()[0]
MEDIDA_DESDE = {c: md_row[c] for c in CAPS}

print("Medido desde por capacidad:")
for c in CAPS:
    print(f"  {c:<16} {MEDIDA_DESDE[c]}")

# Categóricos del PDV: NULL -> "Sin Asignar"
for col in ["territorio", "canal"]:
    if col in panel_qa.columns:
        panel_qa = panel_qa.withColumn(col, sf.coalesce(sf.col(col), sf.lit("Sin Asignar")))

print(f"\nPanel de capacidades:")
print(f"  Filas: {panel_qa.count():,} | PDVs únicos: {panel_qa.select('id_cliente').distinct().count():,}")

Universo oficial: canal Tradicional, según la clasificación de canal que trae el `PanelCapacidades`. Todos los análisis y el modelo aplican a este universo; los efectos estimados no se extrapolan a los clientes que quedan fuera.

In [ ]:
n0_filas = panel_qa.count()
n0_pdvs  = panel_qa.select("id_cliente").distinct().count()

panel_qa = panel_qa.filter(sf.col("clasificacion_canal") == "Tradicional").persist(StorageLevel.MEMORY_AND_DISK)

n1_filas = panel_qa.count()
n1_pdvs  = panel_qa.select("id_cliente").distinct().count()

print(f"Universo: canal Tradicional")
print(f"  PDVs:  {n1_pdvs:,} de {n0_pdvs:,} ({n1_pdvs/n0_pdvs*100:.1f}%)")
print(f"  Filas: {n1_filas:,} de {n0_filas:,} ({n1_filas/n0_filas*100:.1f}%)")

In [ ]:
if USE_MUESTRA:
    # Muestra estable de PDVs (semilla fija) para iterar rápido
    pdvs_sample = (panel_qa.select("id_cliente").distinct()
        .sample(fraction=PCT_MUESTRA, seed=42)
        .persist(StorageLevel.MEMORY_AND_DISK))
    pdvs_sample.count()

    panel_qa = panel_qa.join(pdvs_sample, on="id_cliente", how="inner").persist(StorageLevel.MEMORY_AND_DISK)

    n_pdvs = panel_qa.select("id_cliente").distinct().count()
    n_filas = panel_qa.count()
    print(f"Muestra aplicada | PDVs: {n_pdvs:,} | Filas: {n_filas:,}")

In [ ]:
# Panel diagnóstico: grilla mensual balanceada por PDV (para E0 intermitencia y pre-trends)
# Rango temporal y atributos por PDV
rango_pdv = panel_qa.groupBy("id_cliente").agg(
    sf.min("periodo").alias("primer_mes"),
    sf.max("periodo").alias("ultimo_mes"))

catalogo_pdvs = panel_qa.groupBy("id_cliente").agg(
    sf.first("bu", ignorenulls=True).alias("bu"),
    sf.first("territorio", ignorenulls=True).alias("territorio"),
    sf.first("canal", ignorenulls=True).alias("canal"),
    sf.first("subcanal", ignorenulls=True).alias("subcanal"),
    sf.first("tamano_cliente", ignorenulls=True).alias("tamano_cliente"))

# Grilla completa primer_mes..ultimo_mes por PDV, con atributos
grilla = (rango_pdv
    .withColumn("periodo", sf.explode(sf.expr("sequence(primer_mes, ultimo_mes, interval 1 month)")))
    .select("id_cliente", "periodo")
    .join(catalogo_pdvs, on="id_cliente", how="left"))

# Pegar métricas del panel real y marcar las filas que eran gap
cols_atributos = ["bu", "territorio", "canal", "subcanal", "tamano_cliente"]
panel_diagnostico = (grilla
    .join(panel_qa.drop(*cols_atributos), on=["id_cliente", "periodo"], how="left")
    .withColumn("es_gap_relleno", sf.col(OUTCOME).isNull()))

# Rellenar métricas y capacidades de los gaps con 0
cols_metricas = [
    "ingreso_neto_total", "unit_cases_total",
    "ingreso_neto_core", "unit_cases_core",
    "ingreso_neto_online_core", "ingreso_neto_offline_core",
    "unit_cases_online_core", "unit_cases_offline_core",
    "ingreso_neto_multi", "unit_cases_multi",
    "ingreso_neto_online_multi", "ingreso_neto_offline_multi",
    "unit_cases_online_multi", "unit_cases_offline_multi",
    "ingreso_neto_online", "ingreso_neto_offline",
    "unit_cases_online", "unit_cases_offline",
]
cols_a_rellenar = [c for c in cols_metricas + CAPS if c in panel_diagnostico.columns]
for col in cols_a_rellenar:
    panel_diagnostico = panel_diagnostico.withColumn(col, sf.coalesce(sf.col(col), sf.lit(0)))

panel_diagnostico = panel_diagnostico.persist(StorageLevel.MEMORY_AND_DISK)

stats = panel_diagnostico.agg(
    sf.count("*").alias("total"),
    sf.sum(sf.col("es_gap_relleno").cast("int")).alias("gaps")).collect()[0]
pdvs_diag = panel_diagnostico.select("id_cliente").distinct().count()

print(f"Panel diagnóstico:")
print(f"  Filas totales:      {stats['total']:,}")
print(f"  Filas reales:       {stats['total'] - stats['gaps']:,}")
print(f"  Filas rellenadas:   {stats['gaps']:,} ({stats['gaps']/stats['total']*100:.1f}%)")
print(f"  PDVs:               {pdvs_diag:,}")
print(f"  Meses promedio/PDV: {stats['total']/pdvs_diag:.1f}")

In [ ]:
panel_modelo = panel_qa.persist(StorageLevel.MEMORY_AND_DISK)

filas_mod = panel_modelo.count()
pdvs_mod = panel_modelo.select("id_cliente").distinct().count()

print(f"Panel modelo:")
print(f"  Filas: {filas_mod:,} | PDVs: {pdvs_mod:,} | Meses promedio/PDV: {filas_mod/pdvs_mod:.1f}")

# E0. Diagnóstico de intermitencia de PDVs

In [ ]:
# Frecuencia de compra core por PDV
df_int = panel_diagnostico.withColumn("compro", sf.when(sf.col(OUTCOME) > 0, 1).otherwise(0))

resumen_pdv = (df_int.groupBy("id_cliente").agg(
        sf.count("compro").alias("meses_total"),
        sf.sum("compro").alias("meses_activo"),
        sf.first("canal", ignorenulls=True).alias("canal"),
        sf.first("tamano_cliente", ignorenulls=True).alias("tamano"))
    .withColumn("pct_activo", sf.col("meses_activo") / sf.col("meses_total"))
    .persist(StorageLevel.MEMORY_AND_DISK))

medianas = resumen_pdv.approxQuantile(["meses_activo", "meses_total"], [0.5], 0.01)
mediana_activos, mediana_total = medianas[0][0], medianas[1][0]

frec = resumen_pdv.agg(
    sf.count("*").alias("n"),
    sf.sum((sf.col("pct_activo") == 1).cast("int")).alias("completos"),
    sf.sum((sf.col("pct_activo") > 0.8).cast("int")).alias("alto"),
    sf.sum((sf.col("pct_activo") < 0.5).cast("int")).alias("bajo")).collect()[0]

print("Frecuencia de compra core por PDV")
print(f"  Mediana de meses activos:         {mediana_activos:.0f} de {mediana_total:.0f}")
print(f"  PDVs que compran todos los meses: {frec['completos']:,} ({frec['completos']/frec['n']*100:.1f}%)")
print(f"  PDVs con >80% meses activos:      {frec['alto']:,} ({frec['alto']/frec['n']*100:.1f}%)")
print(f"  PDVs con <50% meses activos:      {frec['bajo']:,} ({frec['bajo']/frec['n']*100:.1f}%)")

In [ ]:
# Gap máximo consecutivo sin compra core por PDV
w_pdv_orden = Window.partitionBy("id_cliente").orderBy("periodo")
df_gaps = df_int.withColumn("grupo_run",
    sf.sum(sf.when(sf.col("compro") != sf.lag("compro", 1).over(w_pdv_orden), 1).otherwise(0)).over(w_pdv_orden))

gaps_por_pdv = (df_gaps.filter(sf.col("compro") == 0)
    .groupBy("id_cliente", "grupo_run").agg(sf.count("*").alias("largo_gap"))
    .groupBy("id_cliente").agg(sf.max("largo_gap").alias("max_gap")))

gaps_completos = (resumen_pdv.select("id_cliente")
    .join(gaps_por_pdv, on="id_cliente", how="left")
    .withColumn("max_gap", sf.coalesce(sf.col("max_gap"), sf.lit(0)))
    .persist(StorageLevel.MEMORY_AND_DISK))

print("\nDistribución de gap máximo consecutivo:")
gaps_completos.groupBy("max_gap").count().orderBy("max_gap").show(15)

In [ ]:
resumen_pdv_pd = resumen_pdv.toPandas()
gaps_pd = gaps_completos.toPandas()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("E0. Análisis de intermitencia (sobre venta core)")

# Frecuencia de compra por PDV
axes[0].hist(resumen_pdv_pd["pct_activo"], bins=20, color=COL["serie"][0], alpha=0.85, edgecolor="white")
axes[0].set_title("Frecuencia de compra core por PDV")
axes[0].set_xlabel("% meses con compra core")
axes[0].set_ylabel("Cantidad de PDVs")
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
mediana_pct = resumen_pdv_pd["pct_activo"].median()
axes[0].axvline(mediana_pct, color=COL["alert"], linewidth=1.5, linestyle="--", label=f"Mediana: {mediana_pct:.0%}")
axes[0].legend()
axes[0].grid(axis="y")

# Gap maximo consecutivo
gap_counts = gaps_pd["max_gap"].value_counts().sort_index().head(13)
axes[1].bar(gap_counts.index, gap_counts.values, color=COL["serie"][1], alpha=0.85, edgecolor="white")
axes[1].set_title("Gap máximo consecutivo por PDV")
axes[1].set_xlabel("Meses consecutivos sin compra core")
axes[1].set_ylabel("Cantidad de PDVs")
for umbral, color, label in [(3, COL["alert"], "Umbral (3)"), (4, COL["warn"], "Alt. (4)"), (6, COL["ok"], "Alt. (6)")]:
    axes[1].axvline(umbral - 0.5, color=color, linewidth=1.5, linestyle="--", label=label)
axes[1].legend()
axes[1].grid(axis="y")

# Media de frecuencia por canal
pct_activo_canal = resumen_pdv_pd.groupby("canal")["pct_activo"].mean().sort_values(ascending=True)
axes[2].barh(pct_activo_canal.index, pct_activo_canal.values, color=COL["serie"][2], alpha=0.85, edgecolor="white")
axes[2].set_title("Media de frecuencia de compra por canal")
axes[2].set_xlabel("% meses activos (media)")
axes[2].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[2].grid(axis="x")

plt.tight_layout()
plt.show()

In [ ]:
# Categorias (orden de evaluacion = prioridad, gana la primera):
#   marginal:              compra <6 (FE inservible) o ventana >=12 con compra <12 (tuvo el ano y no compro)
#   estacional:            >=2 gaps que arrancan en <=2 meses calendario; va antes del corte para no cortar una temporada
#   gap_revenue_distinto:  gap mayor a 6 meses y revenue post/pre fuera de [0.5, 2.0]
#   gap_revenue_similar:   gap mayor a 6 meses y revenue post/pre dentro de [0.5, 2.0]
#   continuo:              gap maximo <=3 meses (incluye truncados)
#   con_gap:               resto, gap maximo entre 4 y 6 meses

print("Categorizacion de PDVs")
panel_inicio, panel_fin = panel_diagnostico.agg(sf.min("periodo"), sf.max("periodo")).collect()[0]
meses_panel_total = ((panel_fin.year - panel_inicio.year) * 12 + (panel_fin.month - panel_inicio.month) + 1)
print(f"\nPanel: {panel_inicio} a {panel_fin} ({meses_panel_total} meses)")

compras_pdv = panel_diagnostico.filter(sf.col("ingreso_neto_core") > 0)
metricas_basicas = (compras_pdv.groupBy("id_cliente")
    .agg(
        sf.count("*").alias("meses_con_compra"),
        sf.min("periodo").alias("primer_mes_compra"),
        sf.max("periodo").alias("ultimo_mes_compra"))
    .withColumn("largo_vida_meses",
        ((sf.year("ultimo_mes_compra") - sf.year("primer_mes_compra")) * 12 +
         (sf.month("ultimo_mes_compra") - sf.month("primer_mes_compra")) + 1).cast("int"))
    .withColumn("pct_actividad", sf.col("meses_con_compra") / sf.col("largo_vida_meses") * 100))

w_pdv_orden = Window.partitionBy("id_cliente").orderBy("periodo")
df_compro = panel_diagnostico.withColumn("compro", sf.when(sf.col("ingreso_neto_core") > 0, 1).otherwise(0))
df_grupos = df_compro.withColumn("grupo_run",
    sf.sum(sf.when(sf.col("compro") != sf.lag("compro", 1).over(w_pdv_orden), 1).otherwise(0)).over(w_pdv_orden))

gaps_por_pdv = (df_grupos.filter(sf.col("compro") == 0)
    .groupBy("id_cliente", "grupo_run").agg(sf.count("*").alias("largo_gap"))
    .groupBy("id_cliente").agg(sf.max("largo_gap").alias("gap_max")))

gap_info = (df_grupos.filter(sf.col("compro") == 0)
    .groupBy("id_cliente", "grupo_run").agg(
        sf.count("*").alias("largo_gap"),
        sf.min("periodo").alias("inicio_gap"),
        sf.max("periodo").alias("fin_gap")))

# Gap dominante para el chequeo de revenue: el mas largo entre los que superan 6 meses
w_gap_orden = Window.partitionBy("id_cliente").orderBy(sf.col("largo_gap").desc())
gap_corte = (gap_info.filter(sf.col("largo_gap") > 6)
    .withColumn("rn", sf.row_number().over(w_gap_orden))
    .filter(sf.col("rn") == 1)
    .select("id_cliente", "inicio_gap", "fin_gap"))

revenue_pre_gap = (compras_pdv.join(gap_corte, on="id_cliente", how="inner")
    .filter(sf.col("periodo") < sf.col("inicio_gap"))
    .groupBy("id_cliente").agg(sf.avg("ingreso_neto_core").alias("revenue_pre_gap")))
revenue_post_gap = (compras_pdv.join(gap_corte, on="id_cliente", how="inner")
    .filter(sf.col("periodo") > sf.col("fin_gap"))
    .groupBy("id_cliente").agg(sf.avg("ingreso_neto_core").alias("revenue_post_gap")))

# Gaps de 2+ meses y en cuantos meses calendario distintos arrancan, para estacional
gaps_estacional = (gap_info.filter(sf.col("largo_gap") >= 2)
    .withColumn("mes_inicio", sf.month("inicio_gap"))
    .groupBy("id_cliente").agg(
        sf.count("*").alias("n_gaps_total"),
        sf.countDistinct("mes_inicio").alias("n_meses_distintos_inicio")))

metricas_completas = (metricas_basicas
    .join(gaps_por_pdv, on="id_cliente", how="left")
    .join(revenue_pre_gap, on="id_cliente", how="left")
    .join(revenue_post_gap, on="id_cliente", how="left")
    .join(gaps_estacional, on="id_cliente", how="left")
    .fillna(0, subset=["gap_max", "n_gaps_total", "n_meses_distintos_inicio"])
    .withColumn("ratio_post_pre",
        sf.when(sf.col("revenue_pre_gap").isNotNull() & (sf.col("revenue_pre_gap") > 0),
                sf.col("revenue_post_gap") / sf.col("revenue_pre_gap")).otherwise(None)))

mediana_panel = pd.Timestamp(panel_inicio) + pd.Timedelta(days=meses_panel_total * 30 / 2)

# Hay corte solo si existe un gap mayor a 6 meses con compras antes y despues
tiene_corte = sf.col("ratio_post_pre").isNotNull()

# Marginal en dos partes: rescata al entrante reciente (ventana <12) con compra limpia
es_marginal = ((sf.col("meses_con_compra") < 6) |
               ((sf.col("largo_vida_meses") >= 6) & (sf.col("meses_con_compra") < 6)))

categorias = (metricas_completas.withColumn("categoria",
        sf.when(es_marginal, "marginal")
        .when((sf.col("n_gaps_total") >= 2) &
              (sf.col("n_meses_distintos_inicio") <= 2) &
              (sf.col("pct_actividad") >= 50), "estacional")
        .when(tiene_corte & ((sf.col("ratio_post_pre") < 0.5) | (sf.col("ratio_post_pre") > 2.0)), "gap_revenue_distinto")
        .when(tiene_corte, "gap_revenue_similar")
        .when(sf.col("gap_max") <= 3, "continuo")
        .otherwise("con_gap"))
    .withColumn("es_truncado_inicio",
        (sf.col("ultimo_mes_compra") < sf.lit(mediana_panel).cast("date")) & (sf.col("pct_actividad") >= 80))
    .withColumn("es_truncado_final",
        (sf.col("primer_mes_compra") > sf.lit(mediana_panel).cast("date")) & (sf.col("pct_actividad") >= 80))
    .persist(StorageLevel.MEMORY_AND_DISK))

n_categorias = categorias.count()
print(f"\nPDVs clasificados: {n_categorias:,}")

resumen_cat = (categorias.groupBy("categoria")
    .agg(sf.count("*").alias("n_pdvs"))
    .withColumn("pct_pdvs", sf.col("n_pdvs") / n_categorias * 100)
    .orderBy(sf.desc("n_pdvs")).toPandas())
resumen_cat["n_pdvs_fmt"] = resumen_cat["n_pdvs"].apply(lambda x: f"{x:,}")
resumen_cat["pct_pdvs_fmt"] = resumen_cat["pct_pdvs"].apply(lambda x: f"{x:.2f}%")
print("\nDistribucion por categoria:")
print(resumen_cat[["categoria", "n_pdvs_fmt", "pct_pdvs_fmt"]].to_string(index=False))

In [ ]:
print(f"Cobertura por capacidad y categoría - {BU}")
panel_con_cat = (panel_modelo
    .join(categorias.select("id_cliente", "categoria"), on="id_cliente", how="inner")
    .persist(StorageLevel.MEMORY_AND_DISK))

capacidades_eval = list(CAPACIDADES_CONTINUAS) + list(CAPACIDADES_BINARIAS)
print(f"\nCapacidades continuas: {list(CAPACIDADES_CONTINUAS)}")
print(f"Capacidades binarias:  {list(CAPACIDADES_BINARIAS)}")
print(f"Total a evaluar:       {len(capacidades_eval)}")

# Activa: continuas si valor > 0, binarias si valor == 1. Conteo por categoría en una sola pasada.
exprs_activa = {c: sf.when(sf.col(c) > 0, 1).otherwise(0) for c in CAPACIDADES_CONTINUAS}
exprs_activa.update({c: sf.when(sf.col(c) == 1, 1).otherwise(0) for c in CAPACIDADES_BINARIAS})

agg_exprs = [sf.count("*").alias("filas_total")]
agg_exprs += [sf.sum(exprs_activa[c]).alias(f"_act_{c}") for c in capacidades_eval]

cobertura = (panel_con_cat
    .groupBy("categoria")
    .agg(*agg_exprs)
    .orderBy(sf.desc("filas_total"))
    .toPandas())

n_pdvs_total = panel_con_cat.select("id_cliente").distinct().count()
n_filas_total = int(cobertura["filas_total"].sum())
print(f"\nPanel sin E0: {n_pdvs_total:,} PDVs, {n_filas_total:,} filas")

# Fila TOTAL sumando los conteos ya calculados
total_row = {"categoria": "TOTAL", "filas_total": n_filas_total}
for c in capacidades_eval:
    total_row[f"_act_{c}"] = int(cobertura[f"_act_{c}"].sum())
cobertura = pd.concat([cobertura, pd.DataFrame([total_row])], ignore_index=True)

# Encabezados cortos y únicos para que la tabla no se corte en el print
abrev, usados = {}, set()
for c in capacidades_eval:
    a, n = c[:4], 4
    while a in usados:
        n += 1
        a = c[:n]
    usados.add(a)
    abrev[c] = a

for c in capacidades_eval:
    cobertura[abrev[c]] = (cobertura[f"_act_{c}"] / cobertura["filas_total"] * 100).round(1)

display_df = cobertura[["categoria", "filas_total"] + [abrev[c] for c in capacidades_eval]].copy()
display_df["filas_total"] = display_df["filas_total"].apply(lambda x: f"{x:,}")

print("\n% PDV-meses con capacidad activa por categoría")
print("Abreviaturas: " + "  ".join(f"{abrev[c]}={c}" for c in capacidades_eval) + "\n")
print(display_df.to_string(index=False))

In [ ]:
# Status de cada PDV por capacidad: never (nunca activa), always (siempre), switcher (cambia).
# Los switchers identifican el coeficiente en los modelos
print(f"Status por capacidad y categoría (never / switcher / always) - {BU}")

capacidades_eval = list(CAPACIDADES_CONTINUAS) + list(CAPACIDADES_BINARIAS)
exprs_activa = {c: sf.when(sf.col(c) > 0, 1).otherwise(0) for c in CAPACIDADES_CONTINUAS}
exprs_activa.update({c: sf.when(sf.col(c) == 1, 1).otherwise(0) for c in CAPACIDADES_BINARIAS})

agg_exprs = [sf.count("*").alias("meses_total")]
agg_exprs += [sf.sum(exprs_activa[c]).alias(f"meses_activo_{c}") for c in capacidades_eval]

pdv_meses = (panel_con_cat
    .groupBy("id_cliente", "categoria")
    .agg(*agg_exprs))

for c in capacidades_eval:
    pdv_meses = pdv_meses.withColumn(f"status_{c}",
        sf.when(sf.col(f"meses_activo_{c}") == 0, "never")
         .when(sf.col(f"meses_activo_{c}") == sf.col("meses_total"), "always")
         .otherwise("switcher"))

# Contar PDVs por status y categoría en una sola pasada
agg_status = [sf.count("*").alias("n_pdvs")]
for c in capacidades_eval:
    agg_status += [
        sf.sum(sf.when(sf.col(f"status_{c}") == "never", 1).otherwise(0)).alias(f"{c}_never_n"),
        sf.sum(sf.when(sf.col(f"status_{c}") == "switcher", 1).otherwise(0)).alias(f"{c}_switch_n"),
        sf.sum(sf.when(sf.col(f"status_{c}") == "always", 1).otherwise(0)).alias(f"{c}_always_n")]

resumen = (pdv_meses
    .groupBy("categoria")
    .agg(*agg_status)
    .toPandas())

# Fila TOTAL sumando los conteos ya calculados
total_row = {"categoria": "TOTAL", "n_pdvs": int(resumen["n_pdvs"].sum())}
for c in capacidades_eval:
    for s in ["never_n", "switch_n", "always_n"]:
        total_row[f"{c}_{s}"] = int(resumen[f"{c}_{s}"].sum())
resumen = pd.concat([resumen, pd.DataFrame([total_row])], ignore_index=True)

for c in capacidades_eval:
    resumen[f"{c}_never"]  = (resumen[f"{c}_never_n"]  / resumen["n_pdvs"] * 100).round(1)
    resumen[f"{c}_switch"] = (resumen[f"{c}_switch_n"] / resumen["n_pdvs"] * 100).round(1)
    resumen[f"{c}_always"] = (resumen[f"{c}_always_n"] / resumen["n_pdvs"] * 100).round(1)

# Orden de lectura: mantener, cortar/gap, excluir, total
orden_cat = ['continuo_completo', 'truncado_inicio', 'truncado_final',
    'continuo_gap_aislado', 'estacional', 'disperso', 'marginal',
    'dos_vidas_revenue_distinto', 'dos_vidas_revenue_similar', 'TOTAL']
resumen["_orden"] = resumen["categoria"].apply(lambda x: orden_cat.index(x) if x in orden_cat else 999)
resumen = resumen.sort_values("_orden").drop(columns="_orden").reset_index(drop=True)

# Una tabla por capacidad
for c in capacidades_eval:
    df_show = resumen[["categoria", "n_pdvs", f"{c}_never", f"{c}_switch", f"{c}_always"]].copy()
    df_show.columns = ["categoria", "n_pdvs", "%never", "%switch", "%always"]
    df_show["n_pdvs"] = df_show["n_pdvs"].apply(lambda x: f"{x:,}")
    print(f"\n{c}")
    print(df_show.to_string(index=False))

In [ ]:
# Distribución temporal por categoría: en qué años viven los PDVs de cada categoría.
# Cruzado con el status por capacidad: si una categoría es never-treated en X y vive antes de que X exista, es contrafactual histórico; si vive cuando X ya existe y no la adopta, es contemporáneo.
print(f"Distribución temporal por categoría - {BU}")

# Agrega el año al panel ya categorizado, sin re-armar el join
panel_year = panel_con_cat.withColumn("year", sf.year("periodo"))

orden_cat = ['continuo_completo', 'truncado_inicio', 'truncado_final',
    'continuo_gap_aislado', 'estacional', 'disperso', 'marginal',
    'dos_vidas_revenue_distinto', 'dos_vidas_revenue_similar', 'TOTAL']

# Filas por categoría y año
distrib = (panel_year
    .groupBy("categoria", "year")
    .agg(sf.count("*").alias("n_filas"))
    .toPandas())

pivot = distrib.pivot(index="categoria", columns="year", values="n_filas").fillna(0)
pivot["total"] = pivot.sum(axis=1)
years_cols = [c for c in pivot.columns if c != "total"]

pct = pivot[years_cols].div(pivot["total"], axis=0) * 100
pct.loc["TOTAL"] = pivot[years_cols].sum() / pivot["total"].sum() * 100
pct = pct.reindex([c for c in orden_cat if c in pct.index])

print("\n% de filas por año, por categoría (suma horizontal = 100%)\n")
print(pct.round(1).to_string())

# PDVs únicos vivos por año y categoría
pdvs_por_year = (panel_year
    .groupBy("categoria", "year")
    .agg(sf.countDistinct("id_cliente").alias("n_pdvs"))
    .toPandas())

pivot_pdvs = pdvs_por_year.pivot(index="categoria", columns="year", values="n_pdvs").fillna(0).astype(int)
# TOTAL = suma por columna: cada PDV está en una sola categoría, la suma no duplica
pivot_pdvs.loc["TOTAL"] = pivot_pdvs.sum(axis=0)
pivot_pdvs = pivot_pdvs.reindex([c for c in orden_cat if c in pivot_pdvs.index])

print("\n\nPDVs únicos vivos por año, por categoría\n")
print(pivot_pdvs.applymap(lambda x: f"{int(x):,}" if pd.notnull(x) else "").to_string())

# Libera el join (categorias queda en memoria)
panel_con_cat.unpersist()

In [ ]:
print("Aplicar reglas de intermitencia")
# Mantener: unidad coherente (continuo, hueco corto, temporada o mismo nivel tras el gap)
# Cortar: gap_revenue_distinto, se conserva el tramo pre-gap (posible cambio de unidad bajo el ID)
# Excluir: marginal, FE no estimable con <12 meses de compra
CATS_MANTENER   = ["continuo", "con_gap", "estacional", "gap_revenue_similar"]
CATS_CORTAR_GAP = ["gap_revenue_distinto"]
CATS_EXCLUIR    = ["marginal"]

conteo = (categorias
    .withColumn("accion",
        sf.when(sf.col("categoria").isin(CATS_MANTENER), "mantener")
         .when(sf.col("categoria").isin(CATS_CORTAR_GAP), "cortar_gap")
         .when(sf.col("categoria").isin(CATS_EXCLUIR), "excluir")
         .otherwise("sin_accion"))
    .groupBy("accion").count().toPandas().set_index("accion")["count"].to_dict())

n_mantener = conteo.get("mantener", 0)
n_cortar   = conteo.get("cortar_gap", 0)
n_excluir  = conteo.get("excluir", 0)
n_sin      = conteo.get("sin_accion", 0)
n_total    = n_mantener + n_cortar + n_excluir + n_sin

print(f"\nDistribucion de acciones (sobre {n_total:,} PDVs):")
print(f"  Mantener completo:       {n_mantener:>10,}  ({n_mantener/n_total*100:>5.2f}%)")
print(f"  Cortar en gap (pre-gap): {n_cortar:>10,}  ({n_cortar/n_total*100:>5.2f}%)")
print(f"  Excluir completo:        {n_excluir:>10,}  ({n_excluir/n_total*100:>5.2f}%)")
if n_sin > 0:
    print(f"  Aviso: {n_sin:,} PDVs sin accion asignada, revisar listas")

pdvs_mantener   = categorias.filter(sf.col("categoria").isin(CATS_MANTENER)).select("id_cliente")
pdvs_cortar_gap = categorias.filter(sf.col("categoria").isin(CATS_CORTAR_GAP)).select("id_cliente")

# Punto de corte: inicio del gap mas largo (mayor a 6 meses) de cada PDV a cortar
df_compro_b = panel_diagnostico.withColumn("compro", sf.when(sf.col("ingreso_neto_core") > 0, 1).otherwise(0))
w_pdv_orden_b = Window.partitionBy("id_cliente").orderBy("periodo")
df_grupos_b = df_compro_b.withColumn("grupo_run",
    sf.sum(sf.when(sf.col("compro") != sf.lag("compro", 1).over(w_pdv_orden_b), 1).otherwise(0)).over(w_pdv_orden_b))

gaps_b = (df_grupos_b
    .join(sf.broadcast(pdvs_cortar_gap), on="id_cliente", how="inner")
    .filter(sf.col("compro") == 0)
    .groupBy("id_cliente", "grupo_run")
    .agg(sf.count("*").alias("largo_gap"), sf.min("periodo").alias("inicio_gap"))
    .filter(sf.col("largo_gap") > 6))

w_gap_max = Window.partitionBy("id_cliente").orderBy(sf.col("largo_gap").desc())
puntos_corte = (gaps_b
    .withColumn("rn", sf.row_number().over(w_gap_max))
    .filter(sf.col("rn") == 1)
    .select("id_cliente", sf.col("inicio_gap").alias("mes_corte"))
    .persist(StorageLevel.MEMORY_AND_DISK))

n_con_corte = puntos_corte.count()
print(f"\nPDVs cortar_gap con punto de corte calculado: {n_con_corte:,} de {n_cortar:,}")

pdvs_mantener_completo = pdvs_mantener.union(pdvs_cortar_gap).select("id_cliente").distinct()

panel_qa_filtrado = (panel_qa
    .join(sf.broadcast(pdvs_mantener_completo), on="id_cliente", how="inner")
    .join(sf.broadcast(puntos_corte), on="id_cliente", how="left")
    .filter(sf.col("mes_corte").isNull() | (sf.col("periodo") < sf.col("mes_corte")))
    .drop("mes_corte"))

# E1. Deflactación de variables monetarias

In [ ]:
# Fuentes de datos para obtener los indices:
#   MEX → INEGI INPC, base 2018=100
#         https://www.inegi.org.mx/temas/inpc/
#   PER → BCRP / INEI, IPC Lima Metropolitana, base Dic-2021=100
#         https://estadisticas.bcrp.gob.pe/estadisticas/series/mensuales/resultados/PN38705PM/html
#   ECU → INEC, IPC Nacional, base 2014=100
#         https://www.ecuadorencifras.gob.ec/indice-de-precios-al-consumidor/
#   ARG → INDEC, IPC Nacional Nivel General, base Dic-2016=100
#         CSV: https://www.indec.gob.ar/ftp/cuadros/economia/serie_ipc_divisiones.csv

INDICES_INFLACION = {
    'MEX': {
        'fuente': 'INEGI - INPC (base 2018=100)',
        'base_periodo': '2026-03-01',
        'base_valor': 145.544,
        'data': [
            ('2021-01-01', 110.210), ('2021-02-01', 110.907), ('2021-03-01', 111.824),
            ('2021-04-01', 112.190), ('2021-05-01', 112.419), ('2021-06-01', 113.018),
            ('2021-07-01', 113.682), ('2021-08-01', 113.899), ('2021-09-01', 114.601),
            ('2021-10-01', 115.561), ('2021-11-01', 116.884), ('2021-12-01', 117.308),
            ('2022-01-01', 118.002), ('2022-02-01', 118.981), ('2022-03-01', 120.159),
            ('2022-04-01', 120.809), ('2022-05-01', 121.022), ('2022-06-01', 122.044),
            ('2022-07-01', 122.948), ('2022-08-01', 123.803), ('2022-09-01', 124.571),
            ('2022-10-01', 125.276), ('2022-11-01', 125.997), ('2022-12-01', 126.478),
            ('2023-01-01', 127.336), ('2023-02-01', 128.046), ('2023-03-01', 128.389),
            ('2023-04-01', 128.363), ('2023-05-01', 128.084), ('2023-06-01', 128.214),
            ('2023-07-01', 128.832), ('2023-08-01', 129.545), ('2023-09-01', 130.120),
            ('2023-10-01', 130.609), ('2023-11-01', 131.445), ('2023-12-01', 132.373),
            ('2024-01-01', 133.555), ('2024-02-01', 133.681), ('2024-03-01', 134.065),
            ('2024-04-01', 134.336), ('2024-05-01', 134.087), ('2024-06-01', 134.594),
            ('2024-07-01', 136.003), ('2024-08-01', 136.013), ('2024-09-01', 136.080),
            ('2024-10-01', 136.828), ('2024-11-01', 137.424), ('2024-12-01', 137.949),
            ('2025-01-01', 138.343), ('2025-02-01', 138.726), ('2025-03-01', 139.161),
            ('2025-04-01', 139.620), ('2025-05-01', 140.012), ('2025-06-01', 140.405),
            ('2025-07-01', 140.780), ('2025-08-01', 140.867), ('2025-09-01', 141.197),
            ('2025-10-01', 141.708), ('2025-11-01', 142.645), ('2025-12-01', 143.042),
            ('2026-01-01', 143.588), ('2026-02-01', 144.307), ('2026-03-01', 145.544),
        ],
    },
    'PER': {
        'fuente': 'BCRP/INEI - IPC Lima Metropolitana (base Dic-2021=100)',
        'base_periodo': '2026-03-01',
        'base_valor': 119.59,
        'data': [
            ('2021-01-01',  94.66), ('2021-02-01',  94.54), ('2021-03-01',  95.33),
            ('2021-04-01',  95.23), ('2021-05-01',  95.49), ('2021-06-01',  95.98),
            ('2021-07-01',  96.95), ('2021-08-01',  97.90), ('2021-09-01',  98.30),
            ('2021-10-01',  98.87), ('2021-11-01',  99.22), ('2021-12-01', 100.00),
            ('2022-01-01', 100.04), ('2022-02-01', 100.35), ('2022-03-01', 101.84),
            ('2022-04-01', 102.82), ('2022-05-01', 103.21), ('2022-06-01', 104.44),
            ('2022-07-01', 105.42), ('2022-08-01', 106.13), ('2022-09-01', 106.68),
            ('2022-10-01', 107.05), ('2022-11-01', 107.60), ('2022-12-01', 108.46),
            ('2023-01-01', 108.70), ('2023-02-01', 109.02), ('2023-03-01', 110.39),
            ('2023-04-01', 111.01), ('2023-05-01', 111.36), ('2023-06-01', 111.19),
            ('2023-07-01', 111.62), ('2023-08-01', 112.04), ('2023-09-01', 112.06),
            ('2023-10-01', 111.70), ('2023-11-01', 111.52), ('2023-12-01', 111.97),
            ('2024-01-01', 111.99), ('2024-02-01', 112.62), ('2024-03-01', 113.75),
            ('2024-04-01', 113.69), ('2024-05-01', 113.59), ('2024-06-01', 113.73),
            ('2024-07-01', 114.00), ('2024-08-01', 114.32), ('2024-09-01', 114.05),
            ('2024-10-01', 113.94), ('2024-11-01', 114.05), ('2024-12-01', 114.17),
            ('2025-01-01', 114.07), ('2025-02-01', 114.28), ('2025-03-01', 115.21),
            ('2025-04-01', 115.57), ('2025-05-01', 115.51), ('2025-06-01', 115.66),
            ('2025-07-01', 115.92), ('2025-08-01', 115.59), ('2025-09-01', 115.60),
            ('2025-10-01', 115.48), ('2025-11-01', 115.61), ('2025-12-01', 115.89),
            ('2026-01-01', 116.01), ('2026-02-01', 116.81), ('2026-03-01', 119.59),
        ],
    },
    'ECU': {
        'fuente': 'INEC - IPC Nacional (base 2014=100)',
        'base_periodo': '2026-03-01',
        'base_valor': 115.26,
        'data': [
            ('2021-01-01', 104.35), ('2021-02-01', 104.44), ('2021-03-01', 104.63),
            ('2021-04-01', 104.99), ('2021-05-01', 105.08), ('2021-06-01', 104.89),
            ('2021-07-01', 105.45), ('2021-08-01', 105.57), ('2021-09-01', 105.58),
            ('2021-10-01', 105.80), ('2021-11-01', 106.18), ('2021-12-01', 106.26),
            ('2022-01-01', 107.02), ('2022-02-01', 107.27), ('2022-03-01', 107.39),
            ('2022-04-01', 108.03), ('2022-05-01', 108.63), ('2022-06-01', 109.34),
            ('2022-07-01', 109.51), ('2022-08-01', 109.54), ('2022-09-01', 109.93),
            ('2022-10-01', 110.06), ('2022-11-01', 110.05), ('2022-12-01', 110.23),
            ('2023-01-01', 110.36), ('2023-02-01', 110.38), ('2023-03-01', 110.45),
            ('2023-04-01', 110.67), ('2023-05-01', 110.77), ('2023-06-01', 111.18),
            ('2023-07-01', 111.78), ('2023-08-01', 112.34), ('2023-09-01', 112.39),
            ('2023-10-01', 112.19), ('2023-11-01', 111.74), ('2023-12-01', 111.72),
            ('2024-01-01', 111.86), ('2024-02-01', 111.96), ('2024-03-01', 112.28),
            ('2024-04-01', 113.71), ('2024-05-01', 113.58), ('2024-06-01', 112.49),
            ('2024-07-01', 113.54), ('2024-08-01', 113.79), ('2024-09-01', 113.99),
            ('2024-10-01', 113.72), ('2024-11-01', 113.42), ('2024-12-01', 112.31),
            ('2025-01-01', 112.14), ('2025-02-01', 112.24), ('2025-03-01', 112.63),
            ('2025-04-01', 112.93), ('2025-05-01', 114.10), ('2025-06-01', 114.16),
            ('2025-07-01', 114.36), ('2025-08-01', 114.71), ('2025-09-01', 114.81),
            ('2025-10-01', 115.13), ('2025-11-01', 114.62), ('2025-12-01', 114.46),
            ('2026-01-01', 114.88), ('2026-02-01', 115.11), ('2026-03-01', 115.26),
        ],
    },
    'ARG': {
        'fuente': 'INDEC - IPC Nacional Nivel General (base Dic-2016=100)',
        'base_periodo': '2026-03-01',
        'base_valor': 11077.0608,
        'data': [
            ('2021-01-01',   401.5071), ('2021-02-01',   415.8595), ('2021-03-01',   435.8657),
            ('2021-04-01',   453.6503), ('2021-05-01',   468.7250), ('2021-06-01',   483.6049),
            ('2021-07-01',   498.0987), ('2021-08-01',   510.3942), ('2021-09-01',   528.4968),
            ('2021-10-01',   547.0802), ('2021-11-01',   560.9184), ('2021-12-01',   582.4575),
            ('2022-01-01',   605.0317), ('2022-02-01',   633.4341), ('2022-03-01',   676.0566),
            ('2022-04-01',   716.9399), ('2022-05-01',   753.1470), ('2022-06-01',   793.0278),
            ('2022-07-01',   851.7610), ('2022-08-01',   911.1316), ('2022-09-01',   967.3076),
            ('2022-10-01',  1028.7060), ('2022-11-01',  1079.2787), ('2022-12-01',  1134.5875),
            ('2023-01-01',  1202.9790), ('2023-02-01',  1282.7091), ('2023-03-01',  1381.1601),
            ('2023-04-01',  1497.2147), ('2023-05-01',  1613.5895), ('2023-06-01',  1709.6115),
            ('2023-07-01',  1818.0838), ('2023-08-01',  2044.2832), ('2023-09-01',  2304.9242),
            ('2023-10-01',  2496.2730), ('2023-11-01',  2816.0628), ('2023-12-01',  3533.1922),
            ('2024-01-01',  4261.5324), ('2024-02-01',  4825.7881), ('2024-03-01',  5357.0929),
            ('2024-04-01',  5830.2271), ('2024-05-01',  6073.7000), ('2024-06-01',  6351.7145),
            ('2024-07-01',  6607.7479), ('2024-08-01',  6883.4412), ('2024-09-01',  7122.2421),
            ('2024-10-01',  7313.9542), ('2024-11-01',  7491.4314), ('2024-12-01',  7694.0075),
            ('2025-01-01',  7864.1257), ('2025-02-01',  8052.9927), ('2025-03-01',  8353.3158),
            ('2025-04-01',  8585.6078), ('2025-05-01',  8714.4871), ('2025-06-01',  8855.5681),
            ('2025-07-01',  9023.9730), ('2025-08-01',  9193.2441), ('2025-09-01',  9384.0922),
            ('2025-10-01',  9603.8623), ('2025-11-01',  9841.3581), ('2025-12-01', 10121.3715),
            ('2026-01-01', 10413.0309), ('2026-02-01', 10714.6255), ('2026-03-01', 11077.0608),
        ],
    },
}

if BU not in INDICES_INFLACION:
    raise ValueError(f"BU '{BU}' sin tabla de inflación. Disponibles: {list(INDICES_INFLACION.keys())}")

config_bu = INDICES_INFLACION[BU]
INPC_BASE = config_bu['base_valor']
inpc_data = config_bu['data']

# Integridad: base_valor debe ser el índice del base_periodo
idx_base = dict(inpc_data).get(config_bu['base_periodo'])
if idx_base is None or abs(idx_base - INPC_BASE) > 1e-6:
    raise ValueError(f"base_valor de {BU} ({INPC_BASE}) no coincide con el índice en {config_bu['base_periodo']} ({idx_base})")

print(f"Deflactando {BU}:")
print(f"  Fuente:        {config_bu['fuente']}")
print(f"  Base período:  {config_bu['base_periodo']}")
print(f"  Base valor:    {INPC_BASE}")
print(f"  Períodos:      {len(inpc_data)} ({inpc_data[0][0]} a {inpc_data[-1][0]})")

# Factor deflactor = base / índice del mes (lleva nominal a pesos constantes del base_periodo)
deflactores_pd = [(d, idx, INPC_BASE / idx) for d, idx in inpc_data]
deflactores = (spark.createDataFrame(deflactores_pd, schema=["periodo_str", "inpc", "factor_deflactor"])
    .withColumn("periodo", sf.to_date("periodo_str", "yyyy-MM-dd"))
    .drop("periodo_str"))

# Join del factor y deflactado de las métricas monetarias presentes
cols_a_deflactar = [c for c in ["ingreso_neto_core", "ingreso_neto_total", "ingreso_neto_multi"]
                    if c in panel_modelo.columns]

panel_modelo = panel_modelo.join(
    sf.broadcast(deflactores.select("periodo", "factor_deflactor")), on="periodo", how="left")
for col in cols_a_deflactar:
    panel_modelo = panel_modelo.withColumn(f"{col}_real", sf.col(col) * sf.col("factor_deflactor"))

panel_modelo = panel_modelo.persist(StorageLevel.MEMORY_AND_DISK)
n_filas = panel_modelo.count()

# Cobertura del deflactor sobre el panel ya materializado (scan en cache)
sin_factor = panel_modelo.filter(sf.col("factor_deflactor").isNull()).count()
if sin_factor > 0:
    meses_sin = sorted(panel_modelo.filter(sf.col("factor_deflactor").isNull())
        .select("periodo").distinct().toPandas()["periodo"].tolist())
    raise ValueError(f"{sin_factor:,} filas sin factor deflactor. Meses faltantes en INDICES_INFLACION['{BU}']: {meses_sin}")

print(f"\nPanel modelo deflactado")
print(f"  Filas:             {n_filas:,}")
print(f"  Columnas creadas:  {[f'{c}_real' for c in cols_a_deflactar]}")

In [ ]:
# Revenue core promedio original vs deflactado por mes
resumen_e1 = (panel_modelo.filter(sf.col("ingreso_neto_core") > 0).groupBy("periodo").agg(
        sf.avg("ingreso_neto_core").alias("ingreso_nominal"),
        sf.avg("ingreso_neto_core_real").alias("ingreso_real"))
    .orderBy("periodo").toPandas())

fig, ax = plt.subplots()
fig.suptitle("E1. Revenue core original vs deflactado")
ax.plot(resumen_e1["periodo"], resumen_e1["ingreso_nominal"],
        color=COL["accent"], linewidth=2, label="Revenue core original")
ax.plot(resumen_e1["periodo"], resumen_e1["ingreso_real"],
        color=COL["primary"], linewidth=2, label="Revenue core deflactado")
ax.fill_between(resumen_e1["periodo"], resumen_e1["ingreso_real"], resumen_e1["ingreso_nominal"],
                alpha=0.15, color=COL["accent"], label="Efecto inflación")
ax.set_ylabel("Revenue core promedio por PDV (meses con compra)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Primera y última fila del resumen ordenado
fila_ini = resumen_e1.iloc[0]
fila_fin = resumen_e1.iloc[-1]
mes_base = pd.Timestamp(config_bu['base_periodo']).strftime('%b-%y')

print(f"Período inicial: {fila_ini['periodo']}")
print(f"Período final:   {fila_fin['periodo']}")

print(f"\nNominal inicial:  ${fila_ini['ingreso_nominal']:,.0f}")
print(f"Nominal final:    ${fila_fin['ingreso_nominal']:,.0f}")
print(f"Crecimiento nominal: {(fila_fin['ingreso_nominal']/fila_ini['ingreso_nominal'] - 1)*100:.1f}%")

print(f"\nReal inicial (en $ de {mes_base}):  ${fila_ini['ingreso_real']:,.0f}")
print(f"Real final:                      ${fila_fin['ingreso_real']:,.0f}")
print(f"Crecimiento real: {(fila_fin['ingreso_real']/fila_ini['ingreso_real'] - 1)*100:.1f}%")

# E2. Tratamiento de meses con venta cero

In [ ]:
# Detección de churn definitivo (regla del marco: 3 ceros consecutivos de core sin retorno).
# Corre sobre el panel diagnóstico (grilla balanceada con gaps), donde los ceros consecutivos son contables.
df_e2 = panel_diagnostico.withColumn("es_cero", sf.when(sf.col("ingreso_neto_core") == 0, 1).otherwise(0))

w_orden = Window.partitionBy("id_cliente").orderBy("periodo")
df_e2 = df_e2.withColumn("grupo_run",
    sf.sum(sf.when(sf.col("es_cero") != sf.lag("es_cero", 1).over(w_orden), 1).otherwise(0)).over(w_orden))

w_grupo = Window.partitionBy("id_cliente", "grupo_run").orderBy("periodo")
df_e2 = df_e2.withColumn("zeros_consec",
    sf.when(sf.col("es_cero") == 1, sf.row_number().over(w_grupo)).otherwise(sf.lit(0)))

tercer_cero = (df_e2.filter(sf.col("zeros_consec") == 3)
    .groupBy("id_cliente").agg(sf.min("periodo").alias("fecha_tercer_cero")))

ultima_compra = (df_e2.filter(sf.col("ingreso_neto_core") > 0)
    .groupBy("id_cliente").agg(sf.max("periodo").alias("ultima_compra")))

# Churn definitivo: ultima compra core en o antes del 3er cero. Se excluye desde el mes siguiente.
cortes = (tercer_cero.join(ultima_compra, on="id_cliente", how="left")
    .filter(sf.col("ultima_compra") <= sf.col("fecha_tercer_cero"))
    .withColumn("fecha_exclusion", sf.add_months(sf.col("fecha_tercer_cero"), 1))
    .select("id_cliente", "fecha_exclusion")
    .persist(StorageLevel.MEMORY_AND_DISK))

total_pdvs    = panel_diagnostico.select("id_cliente").distinct().count()
pdvs_cortados = cortes.count()

print(f"E2 - detección de churn definitivo")
print(f"  Total PDVs en panel:                  {total_pdvs:,}")
print(f"  PDVs con churn definitivo:            {pdvs_cortados:,}  ({pdvs_cortados/total_pdvs*100:.1f}%)")
print(f"  PDVs intactos (incl. intermitentes):  {total_pdvs-pdvs_cortados:,}  ({(total_pdvs-pdvs_cortados)/total_pdvs*100:.1f}%)")

In [ ]:
# Aplicar E2: corte de churn definitivo + filtrado de meses sin core (deja el panel V2 estricto)
panel_e2_in = panel_modelo
filas_pre = panel_e2_in.count()

panel_corte = (panel_e2_in
    .join(cortes, on="id_cliente", how="left")
    .filter(sf.col("fecha_exclusion").isNull() | (sf.col("periodo") < sf.col("fecha_exclusion")))
    .drop("fecha_exclusion")
    .persist(StorageLevel.MEMORY_AND_DISK))

# Conteo del corte y de los meses sin core en una sola pasada
c = panel_corte.agg(
    sf.count("*").alias("post_corte"),
    sf.sum(sf.when(sf.col("ingreso_neto_core") > 0, 1).otherwise(0)).alias("con_core")).collect()[0]
filas_post_corte  = c["post_corte"]
filas_post_filtro = c["con_core"]

# Panel final: solo meses con core > 0 (requisito del outcome log)
panel_modelo = panel_corte.filter(sf.col("ingreso_neto_core") > 0).persist(StorageLevel.MEMORY_AND_DISK)
pdvs_post = panel_modelo.select("id_cliente").distinct().count()

panel_corte.unpersist()
panel_e2_in.unpersist()

print(f"E2 - corte de churn + filtrado de meses sin core")
print(f"  Filas antes:                 {filas_pre:,}")
print(f"  Filas tras corte de churn:   {filas_post_corte:,}  (removidas: {filas_pre-filas_post_corte:,})")
print(f"  Filas tras filtrar sin core: {filas_post_filtro:,}  (ceros: {filas_post_corte-filas_post_filtro:,})")
print(f"\nPanel modelo")
print(f"  Filas:                {filas_post_filtro:,}")
print(f"  PDVs únicos:          {pdvs_post:,}")
print(f"  Meses promedio/PDV:   {filas_post_filtro/pdvs_post:.1f}")

# E3. Tratamiento de valores atípicos (outliers)

In [ ]:
# E3. Trim de outliers por desvio mensual: para cada PDV-mes mide el factor de desvio de
# unit_cases_core contra el promedio del propio PDV y saca el TRIM_PCT mas extremo de la BU.
# Predominan colapsos (distorsionan la identificacion within-PDV); preserva multicategory.

TRIM_PCT     = 0.05    # % de desvios mas extremos por mes a trimear
MULTI_RESCUE = 0.0     # no excluir colapsos con Multicategory > esto (0 = preservar todo multi)

w = Window.partitionBy("id_cliente")
pm = (panel_modelo
      .withColumn("_ref",    sf.avg("unit_cases_core").over(w))
      .withColumn("_ratio",  sf.col("unit_cases_core") / sf.col("_ref"))
      .withColumn("_factor", sf.greatest(sf.col("_ratio"), sf.lit(1.0)/sf.col("_ratio"))))

X_BU = pm.approxQuantile("_factor", [1.0 - TRIM_PCT], 0.001)[0]

flagged    = sf.col("_factor") > X_BU
es_colapso = sf.col("_ratio") < 1.0
rescatado  = flagged & es_colapso & (sf.col("Multicategory") > MULTI_RESCUE)
pm = pm.withColumn("_excluir", flagged & ~rescatado).persist(StorageLevel.MEMORY_AND_DISK)

d = pm.agg(
    sf.count("*").alias("tot"),
    sf.sum(flagged.cast("int")).alias("n_flag"),
    sf.sum(sf.col("_excluir").cast("int")).alias("n_exc"),
    sf.sum((sf.col("_excluir") & ~es_colapso).cast("int")).alias("n_spk"),
    sf.sum((sf.col("_excluir") & es_colapso).cast("int")).alias("n_col"),
    sf.sum(rescatado.cast("int")).alias("n_res"),
).collect()[0]
print(f"X ({BU}): {X_BU:.2f}   (trim del {TRIM_PCT*100:.0f}% mas extremo por PDV-mes)")
print(f"flaggeados (factor > {X_BU:.2f}): {d['n_flag']:,}  ({d['n_flag']/d['tot']*100:.2f}%)")
print(f"  excluidos:              {d['n_exc']:,}  ({d['n_exc']/d['tot']*100:.2f}%)")
print(f"    spikes:               {d['n_spk']:,}")
print(f"    colapsos:             {d['n_col']:,}")
print(f"  no eliminados multi:   {d['n_res']:,}")

# Histograma del ratio en escala lineal: una sola distribucion, coloreada por destino
lo, hi, step = -2.0, 2.0, 0.04
g = (pm.withColumn("_lr", sf.greatest(sf.lit(lo), sf.least(sf.lit(hi), sf.log10("_ratio"))))
       .withColumn("_bin", sf.floor((sf.col("_lr") - lo) / step))
       .groupBy("_bin").agg(sf.count("*").alias("n")).orderBy("_bin").toPandas())
centers = lo + (g["_bin"].values + 0.5) * step
n = g["n"].values
cut = np.log10(X_BU)
col = np.where(np.abs(centers) > cut, COL["alert"], COL["primary"])   # rojo = se trimea, azul = se conserva

fig, ax = plt.subplots()
fig.suptitle("E3. Desvío mensual de unit_cases_core vs promedio del PDV")
for s in (1, -1):
    ax.axvline(s*cut, color=COL["ref"], linestyle="--", linewidth=1)
ax.bar(centers, n, width=step*0.95, color=col, alpha=0.85)
ax.set_title(f"rojo = {TRIM_PCT*100:.0f}% más extremo que se trimea (corte en factor {X_BU:.2f})")
ax.set_xlabel("log10(ratio mes / promedio del PDV)    izquierda = colapso, derecha = spike")
ax.set_ylabel("PDV-mes")
ax.grid(axis="y")
plt.tight_layout()
plt.show()


panel_modelo = (pm.filter(~sf.col("_excluir")).drop("_ref", "_ratio", "_factor", "_excluir").persist(StorageLevel.MEMORY_AND_DISK))
n_post = panel_modelo.count()
pm.unpersist()
print(f"\nPanel tras E3: {n_post:,} filas  (excluidas: {d['n_exc']:,})")

# E4. Análisis exploratorio previo (margen de adopción)

Seis análisis sobre el margen de adopción de cada capacidad (adopción = primer mes con actividad). Los PDVs con fecha de adopción no confiable (censura POS/DS) se excluyen del análisis de su capacidad vía flag.

In [ ]:
DIAGNOSTICO_E4 = {}

# E4. Panel balanceado para el analisis exploratorio de capacidades binarias.
pdvs_universo = panel_modelo.select("id_cliente").distinct()

panel_e4 = panel_diagnostico.join(pdvs_universo, on="id_cliente", how="inner")
panel_e4 = (panel_e4.join(cortes, on="id_cliente", how="left")
            .filter(sf.col("fecha_exclusion").isNull() | (sf.col("periodo") < sf.col("fecha_exclusion")))
            .drop("fecha_exclusion"))
panel_e4 = panel_e4.join(sf.broadcast(deflactores.select("periodo", "factor_deflactor")), on="periodo", how="left")
panel_e4 = panel_e4.withColumn("ingreso_neto_core_real", sf.col("ingreso_neto_core") * sf.col("factor_deflactor"))
panel_e4 = panel_e4.persist(StorageLevel.MEMORY_AND_DISK)

d = panel_e4.agg(
    sf.count("*").alias("filas"),
    sf.countDistinct("id_cliente").alias("pdvs"),
    sf.sum((sf.col("ingreso_neto_core") == 0).cast("int")).alias("ceros"),
).collect()[0]
print(f"Panel E4 balanceado")
print(f"  Filas:            {d['filas']:,}")
print(f"  PDVs:             {d['pdvs']:,}")
print(f"  Filas con ceros:  {d['ceros']:,}  ({d['ceros']/d['filas']*100:.2f}%)")

CAPS_E4 = CAPS

def panel_para(cap):
    flag = FLAGS_CENSURA.get(cap)
    if flag and flag in panel_e4.columns:
        return panel_e4.filter(sf.col(flag) == 0)
    return panel_e4

In [ ]:
# E4.1 Curva de adopción acumulada por cohorte + E4.2 tamaño del never-treated

for cap in CAPS_E4:
    pe = panel_para(cap)
    total_pdvs = pe.select("id_cliente").distinct().count()
    # Fecha de primera activacion por PDV y reparto tratados / never-treated
    fecha_activacion = (pe.filter(sf.col(cap) > 0)
        .groupBy("id_cliente").agg(sf.min("periodo").alias("fecha_activacion")))
    pdvs_tratados = fecha_activacion.count()
    pdvs_never = total_pdvs - pdvs_tratados

    print(f"\n{cap}")
    print(f"PDVs tratados:       {pdvs_tratados:,}  ({pdvs_tratados/total_pdvs*100:.1f}%)")
    print(f"PDVs never-treated:  {pdvs_never:,}  ({pdvs_never/total_pdvs*100:.1f}%)")

    # Curva de adopcion acumulada por cohorte mensual
    cohortes_mes = (fecha_activacion.groupBy("fecha_activacion").count()
        .orderBy("fecha_activacion").toPandas())
    cohortes_mes["pct_acumulado"] = cohortes_mes["count"].cumsum() / pdvs_tratados * 100

    primera = cohortes_mes["fecha_activacion"].min()
    ultima  = cohortes_mes["fecha_activacion"].max()
    meses_adopcion = (ultima.year - primera.year) * 12 + (ultima.month - primera.month)

    if meses_adopcion <= 3:
        clasificacion, color_estado = f"Concentrada ({meses_adopcion} meses), riesgo bajo", COL["ok"]
    elif meses_adopcion <= 12:
        clasificacion, color_estado = f"Moderada ({meses_adopcion} meses), monitorear", COL["warn"]
    else:
        clasificacion, color_estado = f"Dispersa ({meses_adopcion} meses), requiere diagnóstico completo", COL["alert"]

    pct_never = pdvs_never / total_pdvs * 100
    if pct_never >= 30:
        alerta_never = f"Never-treated grande ({pct_never:.1f}%), riesgo acotado"
    elif pct_never >= 10:
        alerta_never = f"Never-treated moderado ({pct_never:.1f}%), monitorear"
    else:
        alerta_never = f"Never-treated pequeño ({pct_never:.1f}%), validar"
    print(f"Clasificación: {clasificacion}")
    print(alerta_never)

    DIAGNOSTICO_E4.setdefault(cap, {})
    DIAGNOSTICO_E4[cap]["e4_dispersion"] = ("concentrada" if meses_adopcion <= 3 else "moderada" if meses_adopcion <= 12 else "dispersa")
    DIAGNOSTICO_E4[cap]["e4_never_treated_pct"] = round(pct_never, 1)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
    fig.suptitle(f"E4.1 y E4.2 Curva de adopción acumulada: {cap}")

    ax1.bar(cohortes_mes["fecha_activacion"], cohortes_mes["count"], color=COL["primary"], alpha=0.85, width=20)
    ax1.set_ylabel("PDVs nuevos por cohorte")
    ax1.grid(axis="y", linestyle="--", alpha=0.3)

    ax2.plot(cohortes_mes["fecha_activacion"], cohortes_mes["pct_acumulado"], color=COL["primary"], linewidth=2, marker="o", markersize=4)
    ax2.fill_between(cohortes_mes["fecha_activacion"], cohortes_mes["pct_acumulado"], alpha=0.15, color=COL["primary"])
    ax2.axhline(50, color=COL["ref"], linestyle="--", alpha=0.5)
    ax2.axhline(90, color=COL["ref"], linestyle="--", alpha=0.5)
    ax2.set_ylabel("% acumulado de PDVs tratados")
    ax2.set_ylim(0, 105)
    ax2.grid(axis="y", linestyle="--", alpha=0.3)

    fig.text(0.5, -0.02,
        f"Primera cohorte: {primera.strftime('%b-%Y')}  |  "
        f"Última cohorte: {ultima.strftime('%b-%Y')}  |  "
        f"Período de adopción: {meses_adopcion} meses\nClasificación: {clasificacion}",
        ha="center", fontsize=10, color=color_estado, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#F9F9F9", alpha=0.8))
    plt.tight_layout()
    plt.show()

In [ ]:
# E4.3 Evolución del revenue por meses desde la adopción
VENTANA_PRE   = 6
VENTANA_POST  = 6
MIN_ADOPTANTES = 500
MIN_NEVER      = 500

for cap in CAPS_E4:
    pe = panel_para(cap)
    print(f"\nE4. {cap}")

    # Evento por PDV: primer mes con la capacidad activa
    fecha_activacion = (pe.filter(sf.col(cap) > 0)
        .groupBy("id_cliente").agg(sf.min("periodo").alias("fecha_activacion")))
    n_adopt = fecha_activacion.count()

    # Never-adopters: nunca activan la capacidad (contrafactual)
    adopt_ids = pe.filter(sf.col(cap) > 0).select("id_cliente").distinct()
    never_ids = (panel_e4.select("id_cliente").distinct()
        .join(adopt_ids, on="id_cliente", how="left_anti"))
    n_never = never_ids.count()
    print(f"  Adoptantes: {n_adopt:,}   Never-adopters: {n_never:,}")

    if n_adopt < MIN_ADOPTANTES or n_never < MIN_NEVER:
        print(f"  Sin masa suficiente (adoptantes o never por debajo del minimo)")
        DIAGNOSTICO_E4.setdefault(cap, {})
        DIAGNOSTICO_E4[cap]["e4_pretrend"] = "sin_masa"
        continue

    # Revenue promedio de never por mes calendario (tendencia del contrafactual)
    rev_never_cal = (panel_e4.join(never_ids, on="id_cliente", how="inner")
        .filter(sf.col(OUTCOME_REAL) > 0)
        .groupBy("periodo").agg(sf.avg(OUTCOME_REAL).alias("rev_never")))

    # Adoptantes alineados por evento, con el revenue de never del mismo mes calendario
    df_rel = (pe.join(fecha_activacion, on="id_cliente", how="inner")
        .withColumn("meses_desde_adopcion",
            ((sf.year("periodo") - sf.year("fecha_activacion")) * 12 +
             (sf.month("periodo") - sf.month("fecha_activacion"))).cast("int"))
        .filter((sf.col("meses_desde_adopcion") >= -VENTANA_PRE) &
                (sf.col("meses_desde_adopcion") <= VENTANA_POST) &
                (sf.col(OUTCOME_REAL) > 0))
        .join(sf.broadcast(rev_never_cal), on="periodo", how="inner"))

    traj = (df_rel.groupBy("meses_desde_adopcion")
        .agg(sf.avg(OUTCOME_REAL).alias("rev_adopt"),
             sf.avg("rev_never").alias("rev_never"))
        .orderBy("meses_desde_adopcion").toPandas())

    # Senal: pendiente del exceso log (adoptantes sobre never) en el pre.
    # Plano = crecen igual (parallel trends OK). Sube = los adoptantes ya venian mejor.
    pre = traj[traj["meses_desde_adopcion"] < 0]
    exceso_pre = np.log(pre["rev_adopt"]) - np.log(pre["rev_never"])
    pend_pre = np.polyfit(pre["meses_desde_adopcion"], exceso_pre, 1)[0]
    nivel = "fuerte" if abs(pend_pre) >= 0.015 else "moderada"
    if abs(pend_pre) < 0.005:
        bandera, color_estado = "Paralelo, sin pre-trend", COL["ok"]
    elif pend_pre > 0:
        bandera = f"Divergencia {nivel} al alza (adoptantes venian mejor)"
        color_estado = COL["alert"] if nivel == "fuerte" else COL["warn"]
    else:
        bandera = f"Divergencia {nivel} a la baja (dip pre-adopcion)"
        color_estado = COL["alert"] if nivel == "fuerte" else COL["warn"]
    print(f"  Exceso pre: {pend_pre*100:+.3f}%/mes  |  {bandera}")

    DIAGNOSTICO_E4.setdefault(cap, {})
    DIAGNOSTICO_E4[cap]["e4_pretrend"] = bandera

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.suptitle(f"E4.3 Pre-trend vs never-adopters: {cap}")
    ax.plot(traj["meses_desde_adopcion"], traj["rev_adopt"],
        color=COL["serie"][0], linewidth=2, marker="o", markersize=5, label="Adoptantes")
    ax.plot(traj["meses_desde_adopcion"], traj["rev_never"],
        color=COL["serie"][1], linewidth=2, marker="s", markersize=5, label="Never-adopters")
    ax.axvline(0, color=COL["alert"], linewidth=1.5, linestyle="--", alpha=0.8)
    ax.axvspan(-VENTANA_PRE, 0, alpha=0.07, color=COL["ref"])
    ax.set_xlabel("Meses desde el primer uso de la capacidad")
    ax.set_ylabel(f"{OUTCOME_REAL} (promedio)")
    ax.set_title(f"exceso pre: {pend_pre*100:+.3f}%/mes  |  {bandera}", fontsize=10, color=color_estado)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax.legend(fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()

print("\nRESUMEN E4")
print(pd.DataFrame([{'Capacidad': c, **v} for c, v in DIAGNOSTICO_E4.items()]).to_string(index=False))

In [ ]:
# E4.4 Comparación de tendencias pre-adopción por cohorte
for cap in CAPS_E4:
    pe = panel_para(cap)
    print(f"\n{cap}")

    fecha_activacion = (pe.filter(sf.col(cap) > 0)
        .groupBy("id_cliente").agg(sf.min("periodo").alias("fecha_activacion")))

    cohortes_tamano = (fecha_activacion.groupBy("fecha_activacion").count()
        .orderBy(sf.col("count").desc()).toPandas())

    top2 = [pd.Timestamp(x).date() for x in cohortes_tamano.head(2)["fecha_activacion"].tolist()]
    fechas_ordenadas = sorted(pd.Timestamp(x).date() for x in cohortes_tamano["fecha_activacion"].tolist())
    mediana_fecha = fechas_ordenadas[len(fechas_ordenadas) // 2]

    lbl_top0 = f"{pd.Timestamp(top2[0]).strftime('%b-%Y')} (top)"
    grupo_col = sf.when(sf.col("fecha_activacion") == sf.lit(top2[0]), sf.lit(lbl_top0))
    if len(top2) > 1:
        lbl_top1 = f"{pd.Timestamp(top2[1]).strftime('%b-%Y')} (top)"
        grupo_col = grupo_col.when(sf.col("fecha_activacion") == sf.lit(top2[1]), sf.lit(lbl_top1))
    grupo_col = (grupo_col
        .when(sf.col("fecha_activacion") < sf.lit(mediana_fecha), sf.lit("Resto tempranas"))
        .otherwise(sf.lit("Resto tardias")))

    df_tratados = (pe.join(fecha_activacion, on="id_cliente", how="inner")
        .withColumn("meses_desde_adopcion",
            ((sf.year("periodo") - sf.year("fecha_activacion")) * 12 +
             (sf.month("periodo") - sf.month("fecha_activacion"))).cast("int"))
        .withColumn("grupo_cohorte", grupo_col)
        .filter((sf.col("meses_desde_adopcion") >= -12) & (sf.col("meses_desde_adopcion") < 0)))

    # Revenue neto promedio deflactado por cohorte en los meses previos, en nivel
    revenue_pre = (df_tratados.groupBy("grupo_cohorte", "meses_desde_adopcion")
        .agg(sf.avg(OUTCOME_REAL).alias("revenue_promedio"))
        .orderBy("grupo_cohorte", "meses_desde_adopcion").toPandas())

    # Trayectorias pre distintas entre cohortes (cambio % en la ventana pre por cohorte)
    piv = revenue_pre.pivot_table(index="grupo_cohorte", columns="meses_desde_adopcion", values="revenue_promedio")
    pend = (piv[piv.columns.max()] - piv[piv.columns.min()]) / piv[piv.columns.min()] * 100
    spread = float(pend.max() - pend.min())
    DIAGNOSTICO_E4.setdefault(cap, {})
    DIAGNOSTICO_E4[cap]["e4_pretrend_cohortes"] = "distintas" if spread > 10 else "similares"
    print(f"  spread de cambio pre entre cohortes: {spread:.1f} pts -> {DIAGNOSTICO_E4[cap]['e4_pretrend_cohortes']}")

    paleta_cohorte = [COL["serie"][i] for i in (0, 2, 4, 1, 3, 5)]
    grupos_orden = sorted(revenue_pre["grupo_cohorte"].unique())
    color_cohorte = {g: paleta_cohorte[i % len(paleta_cohorte)] for i, g in enumerate(grupos_orden)}

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.suptitle(f"E4.4 Tendencias pre-adopcion por cohorte: {cap}")
    for grupo in grupos_orden:
        d = revenue_pre[revenue_pre["grupo_cohorte"] == grupo]
        ax.plot(d["meses_desde_adopcion"], d["revenue_promedio"],
                color=color_cohorte[grupo], linewidth=2, marker="o", markersize=4, label=grupo)
    ax.set_xlabel("Meses antes de la adopcion")
    ax.set_ylabel(f"{OUTCOME_REAL} (promedio deflactado)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax.legend(fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# E4.5 Comparación del revenue post-adopción entre cohortes
for cap in CAPS_E4:
    pe = panel_para(cap)
    print(f"\n{cap}")

    fecha_activacion = (pe.filter(sf.col(cap) > 0)
        .groupBy("id_cliente").agg(sf.min("periodo").alias("fecha_activacion")))

    cohortes_tamano = (fecha_activacion.groupBy("fecha_activacion").count()
        .orderBy(sf.col("count").desc()).toPandas())

    top2 = [pd.Timestamp(x).date() for x in cohortes_tamano.head(2)["fecha_activacion"].tolist()]
    fechas_ordenadas = sorted(pd.Timestamp(x).date() for x in cohortes_tamano["fecha_activacion"].tolist())
    mediana_fecha = fechas_ordenadas[len(fechas_ordenadas) // 2]

    lbl_top0 = f"{pd.Timestamp(top2[0]).strftime('%b-%Y')} (top)"
    grupo_col = sf.when(sf.col("fecha_activacion") == sf.lit(top2[0]), sf.lit(lbl_top0))
    if len(top2) > 1:
        lbl_top1 = f"{pd.Timestamp(top2[1]).strftime('%b-%Y')} (top)"
        grupo_col = grupo_col.when(sf.col("fecha_activacion") == sf.lit(top2[1]), sf.lit(lbl_top1))
    grupo_col = (grupo_col
        .when(sf.col("fecha_activacion") < sf.lit(mediana_fecha), sf.lit("Resto tempranas"))
        .otherwise(sf.lit("Resto tardías")))

    df_tratados = (pe.join(fecha_activacion, on="id_cliente", how="inner")
        .withColumn("meses_desde_adopcion",
            ((sf.year("periodo") - sf.year("fecha_activacion")) * 12 +
             (sf.month("periodo") - sf.month("fecha_activacion"))).cast("int"))
        .withColumn("grupo_cohorte", grupo_col)
        .filter((sf.col("meses_desde_adopcion") >= 0) & (sf.col("meses_desde_adopcion") <= 12))
        .persist(StorageLevel.MEMORY_AND_DISK))

    # Revenue neto promedio deflactado por cohorte, alineadas desde t=0
    revenue_post = (df_tratados.groupBy("grupo_cohorte", "meses_desde_adopcion")
        .agg(sf.avg(OUTCOME_REAL).alias("revenue_promedio"))
        .orderBy("grupo_cohorte", "meses_desde_adopcion").toPandas())

    # Senal: diferencia de nivel post entre cohortes (heterogeneidad estructural)
    rev_grupo = (df_tratados.filter(sf.col("meses_desde_adopcion") <= 6)
        .groupBy("grupo_cohorte").agg(sf.avg(OUTCOME_REAL).alias("revenue_prom")).toPandas())
    if len(rev_grupo) >= 2:
        diff_pct = (rev_grupo["revenue_prom"].max() - rev_grupo["revenue_prom"].min()) / rev_grupo["revenue_prom"].min() * 100
        heterogenea = diff_pct > 50
        DIAGNOSTICO_E4.setdefault(cap, {})
        DIAGNOSTICO_E4[cap]["e4_heterogeneidad"] = "heterogenea" if heterogenea else "homogenea"
        patron = (f"Diferencia entre cohortes: {diff_pct:.1f}%, "
                  + ("cohortes con niveles muy distintos" if heterogenea else "cohortes parejas"))
        color_estado = COL["alert"] if heterogenea else COL["ok"]
        print(patron)
    else:
        patron, color_estado = "Una sola cohorte, sin comparacion", COL["ok"]

    paleta_cohorte = [COL["serie"][i] for i in (0, 2, 4, 1, 3, 5)]
    grupos_orden = sorted(revenue_post["grupo_cohorte"].unique())
    color_cohorte = {g: paleta_cohorte[i % len(paleta_cohorte)] for i, g in enumerate(grupos_orden)}

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.suptitle(f"E4.5 Revenue post-adopción por cohorte: {cap}")
    for grupo in grupos_orden:
        datos = revenue_post[revenue_post["grupo_cohorte"] == grupo]
        if len(datos) > 0:
            ax.plot(datos["meses_desde_adopcion"], datos["revenue_promedio"],
                    color=color_cohorte[grupo], linewidth=2, marker="o", markersize=4, label=grupo)
    ax.axvline(0, color=COL["ref"], linewidth=1, linestyle="--", alpha=0.6)
    ax.set_xlabel("Meses desde adopción")
    ax.set_ylabel(f"{OUTCOME_REAL} (promedio deflactado)")
    ax.set_title(patron, fontsize=10, color=color_estado)
    ax.legend(fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# E4.6 Concentración de la adopción en el mes modal
# Una binaria solo es identificable si los PDVs adoptan en meses distintos.
# Si la mayoría adopta el MISMO mes queda colineal con el FE de tiempo y su
UMBRAL_SINCRONIZADO = 0.70   # modal >= 70% => no identificable

print(f"E4.4 — Concentración del primer mes activo ({BU})")
for cap in CAPS_E4:
    pe = panel_para(cap)
    fa = (pe.filter(sf.col(cap) > 0)
          .groupBy("id_cliente").agg(sf.min("periodo").alias("fa")))
    n_adopt = fa.count()
    if n_adopt == 0:
        continue
    top = fa.groupBy("fa").count().orderBy(sf.col("count").desc()).first()
    modal_pct = top["count"] / n_adopt
    n_meses = fa.select("fa").distinct().count()
    flag = "  <<< SINCRONIZADO: no identificable (colineal con el efecto de período)" if modal_pct >= UMBRAL_SINCRONIZADO else ""
    print(f"  {cap:<16} adoptantes={n_adopt:>9,} | mes modal {pd.Timestamp(top['fa']).strftime('%Y-%m')}: "
          f"{modal_pct*100:5.1f}% | meses distintos: {n_meses}{flag}")
    DIAGNOSTICO_E4.setdefault(cap, {})
    DIAGNOSTICO_E4[cap]["e4_sincronizado"] = "si" if modal_pct >= UMBRAL_SINCRONIZADO else "no"

In [ ]:
print("\nRESUMEN E4")
orden = ["e4_dispersion", "e4_never_treated_pct", "e4_pretrend", "e4_pretrend_cohortes", "e4_heterogeneidad"]
df_res = pd.DataFrame([{"Capacidad": c, **v} for c, v in DIAGNOSTICO_E4.items()])
df_res = df_res[["Capacidad"] + [k for k in orden if k in df_res.columns]]
print(df_res.to_string(index=False))

# E5. Cobertura temporal de las fuentes (ceros falsos)

Detecta fuentes que arrancan después del inicio del panel (señal: el primer mes de la fuente concentra el 30% o más de las primeras activaciones). Regla de tratamiento: las capacidades de stock se retro-completan con su primer valor observado; en los programas, los ceros previos son verdaderos y no se modifican.

In [ ]:
# E5 Detección de arranque tardío artificial por capacidad
# artificial si: 1er_dato > inicio_panel  Y  modal(1er mes activo) = 1er_dato  Y  modal >= 30%
CAP_TIPO = {c: ('stock' if c in CAPACIDADES_STOCK else 'programa') for c in CAPS}
min_panel = panel_modelo.agg(sf.min("periodo")).collect()[0][0]

print(f"E5 — Cobertura temporal de fuentes ({BU}) | inicio panel: {min_panel}")
for cap in CAPS:
    act = panel_modelo.filter(sf.col(cap) > 0)
    fd = act.agg(sf.min("periodo")).collect()[0][0]
    if fd is None:
        continue
    fa = act.groupBy("id_cliente").agg(sf.min("periodo").alias("fa"))
    n = fa.count()
    top = fa.groupBy("fa").count().orderBy(sf.col("count").desc()).first()
    modal_pct = top["count"] / n
    artificial = (fd > min_panel) and (top["fa"] == fd) and (modal_pct >= 0.30)
    if artificial:
        acc = ("RETRO-COMPLETAR cohorte inicial (stock: el activo ya estaba)"
               if CAP_TIPO[cap] == 'stock' else "sin cambio (programa: lanzamiento real, ceros verdaderos)")
        print(f"  {cap:<16} dato desde {fd} | {modal_pct*100:.1f}% adopta ese mes  <<< ARRANQUE TARDÍO ({CAP_TIPO[cap]}) -> {acc}")
    else:
        print(f"  {cap:<16} dato desde {fd} | modal 1er-activo {modal_pct*100:.1f}% -> OK")

In [ ]:
# E5 Retro-completado de la cohorte inicial (solo capacidades de stock)
inicio_panel = panel_modelo.agg(sf.min("periodo")).collect()[0][0]

for cap in [c for c in CAPACIDADES_STOCK if c in panel_modelo.columns]:
    act = panel_modelo.filter(sf.col(cap).isNotNull())
    inicio_fuente = act.agg(sf.min("periodo")).collect()[0][0]
    if inicio_fuente is None or inicio_fuente <= inicio_panel:
        print(f"{cap}: la fuente cubre el panel completo (desde {inicio_fuente}) - sin retro-completado")
        continue

    cohorte_inicial = (act.filter(sf.col("periodo") == sf.lit(inicio_fuente))
                       .groupBy("id_cliente").agg(sf.first(cap).alias("_v0")))
    cond_bf = ((sf.col("periodo") < sf.lit(inicio_fuente))
               & sf.col("_v0").isNotNull()
               & sf.col(cap).isNull())

    panel_modelo = panel_modelo.join(cohorte_inicial, on="id_cliente", how="left")
    n_bf = panel_modelo.filter(cond_bf).count()
    panel_modelo = (panel_modelo
        .withColumn(cap, sf.when(cond_bf, sf.col("_v0")).otherwise(sf.col(cap)))
        .drop("_v0"))

    print(f"{cap}: retro-completado | fuente desde {inicio_fuente} (panel desde {inicio_panel}) | "
          f"cohorte inicial {cohorte_inicial.count():,} PDVs | {n_bf:,} celdas")

panel_modelo = panel_modelo.persist(StorageLevel.MEMORY_AND_DISK)

In [ ]:
# Capacidades sin registro -> 0 (el estado "programa existe, PDV no adopta" queda como control)
for cap in CAPS:
    panel_modelo = panel_modelo.withColumn(cap, sf.coalesce(sf.col(cap), sf.lit(0.0)))
print(f"Capacidades rellenadas a 0 donde no hubo registro: {CAPS}")

# E6. Auditoría de sanidad del panel

E6.1 ventana temporal efectiva · E6.2 huecos de datos · E6.3 cortes abruptos al final · E6.4 valores negativos · E6.5 duplicados por unidad y período (detiene el proceso) · E6.6 rotación anual de la base.

In [ ]:
# E6.1 Ventana efectiva | E6.2 huecos en el panel | E6.3 cortes al final | E6.4 negativos
meses_panel = [r["periodo"] for r in panel_modelo.select("periodo").distinct().orderBy("periodo").collect()]
max_panel = meses_panel[-1]

print(f"E6.1-E6.4 — Auditoría de sanidad ({BU}) | panel: {meses_panel[0]} a {max_panel} ({len(meses_panel)} meses)")
for cap in CAPS:
    act_m = [r["periodo"] for r in panel_modelo.filter(sf.col(cap) > 0)
             .select("periodo").distinct().orderBy("periodo").collect()]
    n_neg = panel_modelo.filter(sf.col(cap) < 0).count()
    if not act_m:
        print(f"  {cap:<16} sin actividad"); continue
    f, l = act_m[0], act_m[-1]
    rango = [m for m in meses_panel if f <= m <= l]
    huecos = sorted(set(rango) - set(act_m))
    corte = len([m for m in meses_panel if m > l])
    flags = []
    if corte:  flags.append(f"CORTE FINAL: sin dato últimos {corte} meses")
    if huecos: flags.append(f"HUECOS: {len(huecos)} ({[str(h)[:7] for h in huecos[:4]]})")
    if n_neg:  flags.append(f"{n_neg} NEGATIVOS")
    print(f"  {cap:<16} dato [{str(f)[:7]} - {str(l)[:7]}] | huecos={len(huecos)} corte_final={corte} neg={n_neg}"
          + ("  <<< " + " | ".join(flags) if flags else "  OK"))

In [ ]:
# E6.2 Interrupciones dentro de la ventana activa de cada PDV
w_pp       = Window.partitionBy("id_cliente").orderBy("periodo")
w_atras    = w_pp.rowsBetween(Window.unboundedPreceding, Window.currentRow)
w_adelante = w_pp.rowsBetween(Window.currentRow, Window.unboundedFollowing)

d_pp = panel_modelo
for c in CAPS:
    obs = sf.when(sf.col(c) > 0, sf.col("periodo"))
    d_pp = (d_pp
        .withColumn(f"_ant_{c}", sf.last(obs, ignorenulls=True).over(w_atras))
        .withColumn(f"_sig_{c}", sf.first(obs, ignorenulls=True).over(w_adelante)))

aggs_pp = []
for c in CAPS:
    dentro = sf.col(f"_ant_{c}").isNotNull() & sf.col(f"_sig_{c}").isNotNull()
    hueco  = dentro & (sf.col(c) == 0)
    aggs_pp += [
        sf.sum(dentro.cast("int")).alias(f"dentro_{c}"),
        sf.sum(hueco.cast("int")).alias(f"hueco_{c}"),
        sf.countDistinct(sf.when(hueco, sf.col("id_cliente"))).alias(f"pdvs_{c}"),
    ]
row_pp = d_pp.agg(*aggs_pp).collect()[0]

print("E6.2 — Interrupciones interiores (PDV-mes en cero con actividad antes y después)")
print(f"{'capacidad':<20}{'PDV-mes en ventana':>20}{'interrumpidos':>15}{'%':>8}{'PDVs':>10}")
for c in CAPS:
    dentro, hueco = row_pp[f"dentro_{c}"] or 0, row_pp[f"hueco_{c}"] or 0
    pct = hueco / dentro * 100 if dentro else 0.0
    print(f"{c:<20}{dentro:>20,}{hueco:>15,}{pct:>7.2f}%{row_pp[f'pdvs_{c}'] or 0:>10,}")

In [ ]:
# E6.5 Duplicados por unidad y período: un duplicado detiene el proceso
n_dup = panel_modelo.groupBy("id_cliente", "periodo").count().filter(sf.col("count") > 1).count()
print(f"E6.5 — Duplicados PDV x mes: {n_dup:,}")
if n_dup > 0:
    raise ValueError(f"Hay {n_dup:,} duplicados PDV x mes: panel mal construido, resolver antes de continuar")

In [ ]:
# E6.6 Rotación anual de la base: altas, bajas y permanencias entre año base y actual
# Cuantifica cuántos clientes salen/entran entre el año base (A) y el actual (B),
# y su tamaño estructural. En la descomposición YoY este efecto es 'composición'
# (el FE de cliente lo separa; no debe atribuirse a capacidades).
maxp   = panel_modelo.agg(sf.max("periodo")).collect()[0][0]
corte_B = (pd.Timestamp(maxp) - pd.DateOffset(months=12)).date()   # inicio año actual
corte_A = (pd.Timestamp(maxp) - pd.DateOffset(months=24)).date()   # inicio año base

base = panel_modelo.select("id_cliente", "periodo", OUTCOME_REAL)
cA = base.filter((sf.col("periodo") > sf.lit(corte_A)) & (sf.col("periodo") <= sf.lit(corte_B))).select("id_cliente").distinct()
cB = base.filter(sf.col("periodo") > sf.lit(corte_B)).select("id_cliente").distinct()

salen  = cA.join(cB, "id_cliente", "left_anti")
entran = cB.join(cA, "id_cliente", "left_anti")
quedan = cA.join(cB, "id_cliente", "inner")

# tamaño estructural: media de log(revenue real) del cliente en toda su historia
tam = (base.withColumn("_ly", sf.log(sf.greatest(sf.col(OUTCOME_REAL), sf.lit(0.01))))
           .groupBy("id_cliente").agg(sf.avg("_ly").alias("_tam")))
m_all = tam.agg(sf.avg("_tam")).collect()[0][0]
def _rel(df, lbl):
    m = df.join(tam, "id_cliente").agg(sf.avg("_tam"), sf.count("*")).collect()[0]
    print(f"  {lbl:<12} {m[1]:>9,} PDVs | tamaño vs promedio: {(np.exp(m[0]-m_all)-1)*100:+.0f}%")
print(f"Rotación de la base — año base (>{corte_A}) vs año actual (>{corte_B}):")
_rel(quedan, "permanecen"); _rel(salen, "salen"); _rel(entran, "entran")

# entrantes 100% nuevos: primera compra histórica dentro del año actual
fp = base.groupBy("id_cliente").agg(sf.min("periodo").alias("_fp"))
n_new = entran.join(fp, "id_cliente").filter(sf.col("_fp") > sf.lit(corte_B)).count()
print(f"  entrantes 100% nuevos (1ra compra en el año actual): {n_new/max(entran.count(),1)*100:.0f}%")

# C5. Tipo de medición y forma funcional (P1-P5)

In [ ]:
# period_id secuencial (insumo de P3, C8 y C9)
if "period_id" not in panel_modelo.columns:
    min_periodo = panel_modelo.agg(sf.min("periodo")).collect()[0][0]
    panel_modelo = panel_modelo.withColumn("period_id",
        ((sf.year("periodo") - sf.year(sf.lit(min_periodo))) * 12 +
         (sf.month("periodo") - sf.month(sf.lit(min_periodo)))).cast("int"))
print("period_id listo")

## P1. Censo de distribución

Distribución del uso sobre la intensidad promedio por PDV (promedio de toda su historia): masa en cero, dispersión entre usuarios (CV, IQR relativo) y concentración por nivel. Clasificación preliminar: Binaria / Continua / Híbrida.

In [ ]:
UMBRAL_CV      = 0.30
UMBRAL_IQR_REL = 0.10

caps_p1 = [c for c in CAPACIDADES_CONTINUAS if c in panel_modelo.columns]
media_pdv = (panel_modelo.groupBy("id_cliente")
             .agg(*[sf.avg(c).alias(c) for c in caps_p1])).toPandas()

resumen_c5 = []
for cap in caps_p1:
    s = media_pdv[cap].fillna(0)
    activos = s[s > 0]
    n_act = len(activos)
    masa_cero = float((s <= 0).mean())
    media = float(activos.mean()) if n_act else float("nan")
    cv = float(activos.std() / media) if n_act and media > 0 else float("nan")
    iqr_rel = float((activos.quantile(0.75) - activos.quantile(0.25)) / media) if n_act and media > 0 else float("nan")

    cumple_cv  = cv >= UMBRAL_CV if cv == cv else False
    cumple_iqr = iqr_rel >= UMBRAL_IQR_REL if iqr_rel == iqr_rel else False
    prelim = "Continua" if (cumple_cv or cumple_iqr) else "Binaria"

    resumen_c5.append({
        "capacidad": cap, "pdvs_activos": n_act, "masa_cero": round(masa_cero, 3),
        "media": round(media, 4) if media == media else None,
        "cv": round(cv, 3) if cv == cv else None,
        "iqr_rel": round(iqr_rel, 3) if iqr_rel == iqr_rel else None,
        "cumple_cv": bool(cumple_cv), "cumple_iqr": bool(cumple_iqr),
        "clasif_preliminar": prelim,
    })

df_c5 = pd.DataFrame(resumen_c5)
print("P1 — Estadísticos de clasificación por capacidad (intensidad promedio por PDV)")
print(df_c5.to_string(index=False))

In [ ]:
# P1 Histogramas de distribución de uso (nivel PDV, activos), con P25/P50/P75
n_caps = len(caps_p1)
fig, axes = plt.subplots(1, n_caps, figsize=(5.5 * n_caps, 4.5))
if n_caps == 1:
    axes = [axes]
fig.suptitle("P1. Distribución del uso por capacidad (promedio histórico por PDV)")
for ax, cap, color in zip(axes, caps_p1, COL["serie"] * 3):
    vals = media_pdv[cap].fillna(0)
    vals = vals[vals > 0]
    if len(vals) == 0:
        ax.set_visible(False)
        continue
    ax.hist(vals, bins=40, color=color, alpha=0.85, edgecolor="white")
    ax.set_yscale("log")
    for q in (0.25, 0.50, 0.75):
        ax.axvline(vals.quantile(q), color=COL["ref"], linewidth=1.2, linestyle="--", alpha=0.7)
    ax.set_title(cap)
    ax.set_xlabel("Nivel de uso")
    ax.set_ylabel("PDVs (log)")
    ax.grid(axis="y")
plt.tight_layout()
plt.show()

## P2. Escala y comparabilidad

Convención común de unidades entre países y capacidades: todas las proporciones en 0-100. Se aplica la conversión y se valida que todas las variables cumplan la convención.

In [ ]:
stats_p2 = panel_modelo.agg(*(
    [sf.expr(f"percentile_approx(`{c}`, 0.99)").alias(f"p99_{c}") for c in CAPS] +
    [sf.min(c).alias(f"min_{c}") for c in CAPS] +
    [sf.max(c).alias(f"max_{c}") for c in CAPS] +
    [sf.avg((sf.col(c) == sf.round(sf.col(c))).cast("int")).alias(f"ent_{c}") for c in CAPS]
)).collect()[0]

PROPORCIONES = []
for c in CAPS:
    p99, ent = stats_p2[f"p99_{c}"], stats_p2[f"ent_{c}"]
    if p99 is None:
        continue
    if p99 <= 1.0001 and (ent or 1) < 0.999:
        PROPORCIONES.append(c)

for cap in PROPORCIONES:
    panel_modelo = panel_modelo.withColumn(cap, sf.col(cap) * sf.lit(100.0))

print(f"Proporciones detectadas: {PROPORCIONES} -> escala 0-100")
print(f"\n{'capacidad':<20}{'min':>10}{'max':>12}  formato")
for c in CAPS:
    mx, mn = stats_p2[f"max_{c}"], stats_p2[f"min_{c}"]
    if mx is None:
        continue
    if c in PROPORCIONES:
        fmt = "proporción 0-100"
    elif (stats_p2[f"ent_{c}"] or 0) >= 0.999 and mx <= 1.0001:
        fmt = "binaria 0/1"
    else:
        fmt = f"conteo 0-{mx:.0f}"
    print(f"{c:<20}{mn:>10.4f}{mx:>12.4f}  {fmt}")

## P3. Dosis-respuesta contra el outcome

Efecto por bins de intensidad dentro de cada PDV, con efectos fijos de PDV y período; los PDVs sin uso son el nivel de referencia. La silueta de la curva es la evidencia de la forma de la relación.

In [ ]:
# P3 Dosis-respuesta within por bins de intensidad, con FE de PDV y periodo
PISO_RATIO = 2.0    # en puntos (0-100): debajo de esto el uso es simbolico, no adopcion real

# Muestra de PDVs; subi fraction si queres mas masa, el cuello es el driver
pdvs_muestra = panel_modelo.select("id_cliente").distinct().sample(fraction=0.90, seed=42)
pdf = (panel_modelo.join(pdvs_muestra, "id_cliente")
       .filter(sf.col(OUTCOME_REAL).isNotNull())
       .withColumn("log_rev", sf.log(sf.col(OUTCOME_REAL) + 1))
       .select("id_cliente", "period_id", "log_rev", *CAPACIDADES_CONTINUAS)
       .toPandas())

print(f"Muestra: {pdf['id_cliente'].nunique():,} PDVs, {len(pdf):,} filas")

n_caps = len(CAPACIDADES_CONTINUAS)
fig, axes = plt.subplots(1, n_caps, figsize=(6 * n_caps, 5))
if n_caps == 1:
    axes = [axes]

fig.suptitle('P3. Dosis-respuesta por bins de intensidad\n(FE de PDV y periodo, HC1; tamano del punto = masa del bin)')

for ax, cap, color in zip(axes, CAPACIDADES_CONTINUAS, COL["serie"]):
    if cap not in pdf.columns:
        ax.set_visible(False)
        continue

    d = pdf[["id_cliente", "period_id", "log_rev", cap]].dropna().copy()
    maxv = d[cap].max()

    # Ratio: nivel 0 = inactivo (incluye uso por debajo del piso); bins de 10% sobre adopcion real.
    # Conteo: nivel entero, 10+ agrupado, nivel 0 = inactivo.
    if cap in PROPORCIONES:
        d["nivel"] = 0
        pos = d[cap] > PISO_RATIO
        d.loc[pos, "nivel"] = (pd.cut(d.loc[pos, cap], bins=np.arange(0, 100.01, 10),
                                      labels=False, include_lowest=True) + 1)
        es_ratio = True
        etiqueta_x = f'{cap} (puntos de uso; <{PISO_RATIO:.0f} = inactivo)'
    else:
        d["nivel"] = d[cap].clip(upper=10).round()
        es_ratio = False
        etiqueta_x = f'{cap} (nivel; 10+ agrupado)'
    d["nivel"] = d["nivel"].astype(int)

    cont = d["nivel"].value_counts()
    niveles_ok = sorted([nv for nv in cont.index if cont[nv] >= 200 and nv != 0])
    if len(niveles_ok) == 0:
        ax.set_title(f'{cap} sin masa por nivel')
        continue
    d = d[d["nivel"].isin([0] + niveles_ok)]

    mod = pf.feols("log_rev ~ i(nivel, ref=0) | id_cliente + period_id",
                   data=d, vcov="HC1")
    coefs = mod.coef()
    ci = mod.confint()

    filas = []
    for nombre in coefs.index:
        m = re.search(r'(\d+)\s*$', str(nombre))
        if m and "nivel" in str(nombre):
            nv = int(m.group(1))
            filas.append((nv, coefs[nombre], ci.loc[nombre].iloc[0], ci.loc[nombre].iloc[1]))
    filas.sort()
    if len(filas) == 0:
        ax.set_title(f'{cap} sin coeficientes')
        continue

    if es_ratio:
        xs = [0.0] + [(f[0] - 0.5) * 10 for f in filas]
    else:
        xs = [0] + [f[0] for f in filas]
    ys = [0.0] + [f[1] for f in filas]
    lo = [0.0] + [f[2] for f in filas]
    hi = [0.0] + [f[3] for f in filas]

    # Tamano del punto segun masa del bin (raiz para comprimir la escala)
    ns = np.array([cont.get(0, 1)] + [cont.get(f[0], 1) for f in filas], dtype=float)
    raiz = np.sqrt(ns)
    sizes = 30 + 320 * (raiz - raiz.min()) / (raiz.max() - raiz.min() + 1e-9)

    ax.fill_between(xs, lo, hi, color=color, alpha=0.18, zorder=1)
    ax.plot(xs, ys, color=color, linewidth=1.3, alpha=0.6, zorder=2)
    ax.scatter(xs, ys, s=sizes, color=color, edgecolor="white", linewidth=0.6, zorder=3)
    ax.axhline(0, color=COL["ref"], linewidth=1, linestyle="--", alpha=0.6)
    ax.set_title(f'{cap}')
    ax.set_xlabel(etiqueta_x)
    ax.set_ylabel('Efecto en log revenue vs inactivo')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## P4. Contraste de formas en el modelo completo

Formas candidatas (lineal, logarítmica, cuadrática) estimadas con efectos fijos sobre la muestra: ajuste (RMSE, R² within), significancia del término adicional y variación within que identifica cada término. A resultados equivalentes se elige la forma más simple.

In [ ]:
EPS = 1
tabla_p4 = []
for cap in caps_p1:
    if cap not in pdf.columns:
        continue
    d = pdf[["id_cliente", "period_id", "log_rev", cap]].dropna().copy()
    if (d[cap] > 0).sum() < 1000:
        continue
    d["_log"] = np.log(d[cap] + EPS)
    d["_sq"]  = (d[cap] ** 2).astype("float64")
    formas = {
        "lineal":     (f"log_rev ~ {cap} | id_cliente + period_id", cap),
        "log":        ("log_rev ~ _log | id_cliente + period_id", "_log"),
        "cuadratica": (f"log_rev ~ {cap} + _sq | id_cliente + period_id", cap),
    }
    for nombre, (fml, t1) in formas.items():
        try:
            m = pf.feols(fml, data=d, vcov="HC1")
        except Exception as e:
            print(f"  {cap} {nombre}: error {str(e)[:60]}")
            continue
        co, pv, ci = m.coef(), m.pvalue(), m.confint()
        fila = {
            "capacidad": cap, "forma": nombre,
            "beta": round(float(co[t1]), 5),
            "ci_low": round(float(ci.loc[t1].iloc[0]), 5),
            "ci_high": round(float(ci.loc[t1].iloc[1]), 5),
            "rmse": round(float(np.sqrt((m.resid() ** 2).mean())), 5),
            "r2_within": round(float(getattr(m, "_r2_within", float("nan"))), 5),
        }
        if nombre == "cuadratica":
            fila["beta_sq"] = round(float(co["_sq"]), 7)
            fila["p_sq"] = round(float(pv["_sq"]), 4)
        tabla_p4.append(fila)

df_p4 = pd.DataFrame(tabla_p4)
print("P4 — Contraste de formas por capacidad")
print(df_p4.to_string(index=False))

In [ ]:
# P4 Variación within / total por capacidad continua (identificación de cada término).
# Fracción de la varianza intra-PDV-período sobre la varianza total: la variación que el TWFE (efectos fijos id_cliente + periodo) usa para identificar el coeficiente continuo.
# Ratio bajo: la variación es entre PDVs, no within, y la capacidad no se sostiene como dosis continua aunque su CV global sea alto. Se mide sobre activos.

UMBRAL_WITHIN = 0.10   # criterio, no del Marco: piso tentativo para sostener forma continua
N_ITER_DEMEAN = 4      # el ratio del demean two-way converge en 2-3 iteraciones

def ratio_within_total(df, col):
    # Activos: PDVs que usan la capacidad alguna vez. Los inactivos solo aportan ceros y hunden el ratio por una razon que no es si la capacidad varia cuando se usa.
    pdvs_activos = df.filter(sf.col(col) > 0).select("id_cliente").distinct()
    d = (df.join(pdvs_activos, on="id_cliente", how="inner")
           .select("id_cliente", "periodo", sf.col(col).cast("double").alias("x")))

    ini = d.agg(
        sf.countDistinct("id_cliente").alias("n_pdvs"),
        sf.avg("x").alias("m"),
        sf.avg(sf.col("x") * sf.col("x")).alias("m2"),
        sf.avg(sf.when(sf.col("x") > 0, sf.col("x"))).alias("media_pos"),
    ).collect()[0]
    var_total = ini["m2"] - ini["m"] ** 2

    # Demean two-way iterativo. localCheckpoint corta el lineage cada iteracion: sin esto la cadena de joins se recomputa bajo desalojo de cache y satura la sesion en panel grande.
    d = d.localCheckpoint(eager=True)
    for _ in range(N_ITER_DEMEAN):
        m_pdv = d.groupBy("id_cliente").agg(sf.avg("x").alias("mb"))
        d = d.join(m_pdv, on="id_cliente").withColumn("x", sf.col("x") - sf.col("mb")).drop("mb")
        m_per = d.groupBy("periodo").agg(sf.avg("x").alias("mb"))
        d = d.join(sf.broadcast(m_per), on="periodo").withColumn("x", sf.col("x") - sf.col("mb")).drop("mb")
        d = d.localCheckpoint(eager=True)

    s = d.agg(sf.avg("x").alias("m"), sf.avg(sf.col("x") * sf.col("x")).alias("m2")).collect()[0]
    var_within = s["m2"] - s["m"] ** 2
    return {'ratio': var_within / var_total if var_total > 0 else 0.0,
            'var_total': var_total, 'var_within': var_within,
            'n_pdvs': ini["n_pdvs"], 'media_pos': ini["media_pos"]}

resumen_within = []
print("C5. Variación within / total por capacidad continua (sobre activos)")
for cap in CAPACIDADES_CONTINUAS:
    if cap not in panel_modelo.columns:
        print(f"\n  '{cap}' no encontrada, se omite")
        continue
    r = ratio_within_total(panel_modelo, cap)
    sd_within = r['var_within'] ** 0.5
    sd_rel = sd_within / r['media_pos'] if r['media_pos'] else 0.0
    forma = "Continua" if r['ratio'] >= UMBRAL_WITHIN else "Presencia (within insuficiente)"

    print(f"\n{cap}")
    print(f"  PDVs activos:        {r['n_pdvs']:>10,}")
    print(f"  Varianza total:      {r['var_total']:>12.5f}")
    print(f"  Varianza within:     {r['var_within']:>12.5f}")
    print(f"  Ratio within/total:  {r['ratio']:>10.4f}  (umbral >={UMBRAL_WITHIN})")
    print(f"  Desvío within:       {sd_within:>10.4f}  ({sd_rel*100:.1f}% de la media activa)")
    print(f"  Clasificación:       {forma}")

    resumen_within.append({'capacidad': cap, 'pdvs_activos': r['n_pdvs'],
        'ratio_within': round(r['ratio'], 4), 'sd_within_rel': round(sd_rel, 3), 'forma': forma})

print("\nRESUMEN within / total (menor within primero)")
print(pd.DataFrame(resumen_within).sort_values('ratio_within').to_string(index=False))

In [ ]:
FORMA, reglas_forma = {}, {}
for cap in CAPACIDADES_MODELO:
    if cap in CAPACIDADES_BINARIAS:
        FORMA[cap] = 'binaria'
        reglas_forma[cap] = 'binaria por tipo de medición'
        continue
    sub = df_p4[df_p4['capacidad'] == cap].set_index('forma')
    if len(sub) < 3:
        FORMA[cap] = 'lineal'
        reglas_forma[cap] = 'sin contraste suficiente -> forma más simple'
        continue

    r_lin, r_log = float(sub.loc['lineal', 'rmse']), float(sub.loc['log', 'rmse'])
    if r_log < r_lin:
        simple, regla = 'log', f'log ordena mejor por RMSE ({(r_lin - r_log) / r_lin * 100:.2f}%)'
    else:
        simple, regla = 'lineal', 'lineal ordena igual o mejor que log -> la más simple'

    act = pdf.loc[pdf[cap] > 0, cap].dropna()
    b1, b2 = float(sub.loc['cuadratica', 'beta']), float(sub.loc['cuadratica', 'beta_sq'])
    p_sq = float(sub.loc['cuadratica', 'p_sq'])
    x_alto = float(act.quantile(0.90))
    peso_curv = abs(b2 * x_alto ** 2) / abs(b1 * x_alto) if b1 != 0 else float('inf')
    vertice = -b1 / (2 * b2) if b2 != 0 else None
    vertice_dentro = vertice is not None and float(act.min()) < vertice < float(act.quantile(0.99))
    material = (peso_curv >= 0.10) or vertice_dentro

    if p_sq < 0.05 and material:
        FORMA[cap] = 'cuadratica'
        motivo = (f'vértice en {vertice:.2f}, dentro del rango' if vertice_dentro
                  else f'curvatura {peso_curv:.0%} del efecto en dosis alta')
        reglas_forma[cap] = f'término cuadrático significativo (p={p_sq:.4f}) y {motivo}'
    else:
        FORMA[cap] = simple
        reglas_forma[cap] = regla

FORMA_OVERRIDE = {}
FORMA.update({k: v for k, v in FORMA_OVERRIDE.items() if k in FORMA})

print(f"{'capacidad':<20}{'forma':<14}regla aplicada")
for c in CAPACIDADES_MODELO:
    marca = '  (override)' if c in FORMA_OVERRIDE else ''
    print(f"{c:<20}{FORMA[c]:<14}{reglas_forma.get(c, '')}{marca}")

## P5. Dinámica temporal (rezagos)

Reglas: las variables de stock no llevan rezago (su persistencia ya está en su valor del mes siguiente); las binarias de estado persistente tampoco (su rezago es casi idéntico al contemporáneo); el resto entra con rezago t-1.

In [ ]:
SIN_REZAGO = set(CAPACIDADES_STOCK) | {'POS', 'DigitalServices'}
REZAGOS = [c for c in CAPACIDADES_MODELO if c not in SIN_REZAGO]

print(f"{'capacidad':<20}{'regla':<38}{'rezago t-1'}")
for c in CAPACIDADES_MODELO:
    if c in CAPACIDADES_STOCK:
        regla = 'stock (persistencia en el propio valor)'
    elif c in SIN_REZAGO:
        regla = 'binaria de estado persistente'
    else:
        regla = 'flujo / intensidad'
    print(f"{c:<20}{regla:<38}{'si' if c in REZAGOS else 'no'}")

# C6. Historia de datos

In [ ]:
# Continuas: (1) historia desde primera actividad >= X meses;
#            (2) >= 20% de PDVs con exposicion real que varian within
# Binarias/ Híbridas:  >= 80% de PDVs tratados en cohortes con >= X pre y >= X post

UMBRAL_MESES_HISTORIA   = 6
UMBRAL_PCT_VARIACION    = 0.20
UMBRAL_PCT_PDVS_BINARIA = 0.80
MIN_MESES_EXPOSICION    = 6
MIN_PRE  = 6
MIN_POST = 3

RANGOS_VARIACION = {
    'Digital':              5.0,
    'Multicategory':        5.0,
    'PedidoSugerido':       5.0,
    'Loyalty':              5.0,
    'Coolers':              1.0,
    'GuidedMissions':       1.0,
    'GuidedMissionsRatio':  5.0,
}

panel_inicio = panel_modelo.agg(sf.min("periodo")).collect()[0][0]
panel_fin    = panel_modelo.agg(sf.max("periodo")).collect()[0][0]
meses_panel  = ((panel_fin.year - panel_inicio.year) * 12 +
                (panel_fin.month - panel_inicio.month) + 1)

print(f"Panel: {panel_inicio} - {panel_fin} ({meses_panel} meses)")

resultados_c6 = []
capacidades_excluidas_por_c6 = []

print(f"\nCONTINUAS: historia >={UMBRAL_MESES_HISTORIA}m | variacion en >={UMBRAL_PCT_VARIACION*100:.0f}% de PDVs con >={MIN_MESES_EXPOSICION} meses activos")

for cap in CAPACIDADES_CONTINUAS:
    if cap not in panel_modelo.columns:
        print(f"\n  '{cap}' no encontrada, se omite")
        continue

    activos = panel_modelo.filter(sf.col(cap) > 0)
    if activos.limit(1).count() == 0:
        print(f"\n{cap} sin actividad en el panel")
        capacidades_excluidas_por_c6.append(cap)
        resultados_c6.append({
            'Capacidad': cap, 'Tipo': 'Continua', '1er mes act': 'NA',
            'Meses historia': 0, 'PDVs expuestos': 0, '% con variacion': 0.0,
            'Resultado': 'Sin actividad'
        })
        continue

    primer_mes = activos.agg(sf.min("periodo")).collect()[0][0]
    meses_historia = ((panel_fin.year - primer_mes.year) * 12 + (panel_fin.month - primer_mes.month) + 1)
    cumple_historia = meses_historia >= UMBRAL_MESES_HISTORIA

    rango_min = RANGOS_VARIACION.get(cap, 0.05)
    pdv_rango = (panel_modelo
        .groupBy("id_cliente").agg(
            sf.max(cap).alias("max_pdv"),
            sf.min(cap).alias("min_pdv"),
            sf.sum((sf.col(cap) > 0).cast("int")).alias("meses_activos"),
        )
        .filter(sf.col("meses_activos") >= MIN_MESES_EXPOSICION)
        .withColumn("rango", sf.col("max_pdv") - sf.col("min_pdv"))
        .withColumn("tiene_variacion", sf.col("rango") >= rango_min)
    )
    n_expuestos = pdv_rango.count()
    n_con_var   = pdv_rango.filter(sf.col("tiene_variacion")).count()
    pct_var     = n_con_var / n_expuestos if n_expuestos > 0 else 0
    cumple_variacion = pct_var >= UMBRAL_PCT_VARIACION

    if cumple_historia and cumple_variacion:
        resultado = 'Cumple'
    elif not cumple_historia and not cumple_variacion:
        resultado = f'No cumple: historia ({meses_historia}m) y variacion ({pct_var*100:.1f}%)'
        capacidades_excluidas_por_c6.append(cap)
    elif not cumple_historia:
        resultado = f'No cumple: historia ({meses_historia}m)'
        capacidades_excluidas_por_c6.append(cap)
    else:
        resultado = f'No cumple: variacion ({pct_var*100:.1f}%)'
        capacidades_excluidas_por_c6.append(cap)

    print(f"\n{cap}")
    print(f"  Primera actividad:              {primer_mes}")
    print(f"  Meses de historia:              {meses_historia}  (umbral >={UMBRAL_MESES_HISTORIA})")
    print(f"  Rango minimo de cambio:         {rango_min}")
    print(f"  PDVs con >={MIN_MESES_EXPOSICION} meses activos:      {n_expuestos:,}")
    print(f"  De esos, con variacion:         {n_con_var:,}  ({pct_var*100:.1f}%)  (umbral >={UMBRAL_PCT_VARIACION*100:.0f}%)")
    print(f"  Resultado:                      {resultado}")

    resultados_c6.append({
        'Capacidad': cap, 'Tipo': 'Continua', '1er mes act': str(primer_mes),
        'Meses historia': meses_historia, 'PDVs expuestos': n_expuestos,
        '% con variacion': round(pct_var * 100, 1), 'Resultado': resultado,
    })

print(f"\nBINARIAS: cohortes con >={MIN_PRE} pre y >={MIN_POST} post; capacidad cumple si >={UMBRAL_PCT_PDVS_BINARIA*100:.0f}% de PDVs OK")

for cap in CAPACIDADES_BINARIAS:
    if cap not in panel_modelo.columns:
        print(f"\n  '{cap}' no encontrada, se omite")
        continue

    fecha_activacion = panel_modelo.filter(sf.col(cap) == 1).groupBy("id_cliente").agg(sf.min("periodo").alias("fecha_activacion"))
    if fecha_activacion.limit(1).count() == 0:
        print(f"\n{cap} sin PDVs tratados")
        capacidades_excluidas_por_c6.append(cap)
        continue

    cohortes = fecha_activacion.groupBy("fecha_activacion").count().orderBy("fecha_activacion").toPandas()

    print(f"\n{cap} por cohorte:")
    print(f"  {'Cohorte':<12} {'PDVs':>8} {'Pre':>5} {'Post':>5}  Resultado")
    print(f"  {'-'*52}")

    pdvs_ok, pdvs_tot = 0, 0
    for _, row in cohortes.iterrows():
        f = row['fecha_activacion']
        n = row['count']
        pre  = (f.year - panel_inicio.year) * 12 + (f.month - panel_inicio.month)
        post = (panel_fin.year - f.year) * 12 + (panel_fin.month - f.month)
        cumple_pre, cumple_post = pre >= MIN_PRE, post >= MIN_POST

        if cumple_pre and cumple_post:
            res = 'Cumple'
            pdvs_ok += n
        elif not cumple_pre:
            res = 'Pre insuficiente'
        else:
            res = 'Post insuficiente'
        pdvs_tot += n
        print(f"  {str(f):<12} {n:>8,} {pre:>5} {post:>5}  {res}")

    pct_pdvs_ok = pdvs_ok / pdvs_tot if pdvs_tot > 0 else 0
    cumple_capacidad = pct_pdvs_ok >= UMBRAL_PCT_PDVS_BINARIA
    if cumple_capacidad:
        res_cap = f'Cumple ({pct_pdvs_ok*100:.1f}% PDVs OK)'
    else:
        res_cap = f'No cumple ({pct_pdvs_ok*100:.1f}% PDVs OK, umbral >={UMBRAL_PCT_PDVS_BINARIA*100:.0f}%)'
        capacidades_excluidas_por_c6.append(cap)

    print(f"\n  PDVs en cohortes que cumplen: {pdvs_ok:,} de {pdvs_tot:,} ({pct_pdvs_ok*100:.1f}%)")
    print(f"  Resultado capacidad:          {res_cap}")

    resultados_c6.append({
        'Capacidad': cap, 'Tipo': 'Binaria', '1er mes act': str(cohortes['fecha_activacion'].min()),
        'Meses historia': meses_panel, 'PDVs expuestos': pdvs_tot,
        '% con variacion': round(pct_pdvs_ok * 100, 1), 'Resultado': res_cap,
    })

print("\nResumen C6")
print(pd.DataFrame(resultados_c6).to_string(index=False))

if capacidades_excluidas_por_c6:
    print(f"\nCapacidades excluidas por C6: {capacidades_excluidas_por_c6}")
    print("No se incluiran en C8 (co-ocurrencia) ni en C9 (combinaciones).")
else:
    print("\nTodas las capacidades cumplen C6.")

In [ ]:
# C6. Diagnostico visual: exposicion de continuas (nuevo) e historia de cada capacidad (timeline original)

# (1) Distribucion de exposicion: meses activos por PDV (continuas)
ma_exprs = [sf.sum((sf.col(c) > 0).cast("int")).alias(c) for c in CAPACIDADES_CONTINUAS]
meses_pdf = panel_modelo.groupBy("id_cliente").agg(*ma_exprs).toPandas()

TRAMOS = [
    ('1 mes', lambda m: m == 1),
    ('2-5',   lambda m: (m >= 2) & (m <= 5)),
    ('6-11',  lambda m: (m >= 6) & (m <= 11)),
    ('12+',   lambda m: m >= 12),
]
dist_exp = []
for cap in CAPACIDADES_CONTINUAS:
    m = meses_pdf[cap]
    m = m[m > 0]
    total = len(m)
    fila = {'Capacidad': cap, 'Activos': total}
    for nombre, cond in TRAMOS:
        fila[nombre] = int(cond(m).sum())
    fila['>=6 meses'] = fila['6-11'] + fila['12+']
    fila['% >=6'] = round(100 * fila['>=6 meses'] / total, 1) if total > 0 else 0.0
    dist_exp.append(fila)
dist_df = pd.DataFrame(dist_exp)
print("C6. Distribucion de exposicion (meses activos por PDV, continuas)")
print(dist_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4.2))
nombres_tramos = [t[0] for t in TRAMOS]
tonos = plt.cm.Blues([0.4, 0.55, 0.72, 0.9])
caps = dist_df['Capacidad'].tolist()
y = list(range(len(caps)))
izq = [0.0] * len(caps)
for j, nombre in enumerate(nombres_tramos):
    props = (dist_df[nombre] / dist_df['Activos'] * 100).tolist()
    ax.barh(y, props, left=izq, color=tonos[j], edgecolor="white", alpha=0.85, label=nombre)
    izq = [a + b for a, b in zip(izq, props)]
for i, cap in enumerate(caps):
    ax.text(102, i, f"{dist_df.loc[i, '% >=6']:.0f}% con >=6m", va="center", ha="left", color=COL["ref"], clip_on=False)
ax.set_yticks(y); ax.set_yticklabels(caps)
ax.set_xlim(0, 100); ax.set_xlabel("% de PDVs activos"); ax.invert_yaxis()
ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.04), frameon=False)
fig.suptitle("C6. Distribucion de exposicion por capacidad.")
plt.subplots_adjust(left=0.14, right=0.80, top=0.84, bottom=0.14)
plt.show()

# (2) Cobertura temporal: desde cuando esta activa cada capacidad (todas)
caps_a_graficar = [r for r in resultados_c6 if r['1er mes act'] != 'NA']
if caps_a_graficar:
    fig, ax = plt.subplots(figsize=(12, max(3.5, 0.55 * len(caps_a_graficar) + 1)))
    for i, r in enumerate(caps_a_graficar):
        primer = pd.to_datetime(r['1er mes act'])
        cumple = r['Resultado'].startswith('Cumple')
        # Pre sin actividad
        ax.barh(i, (primer - pd.Timestamp(panel_inicio)).days, left=pd.Timestamp(panel_inicio),
                height=0.55, color=COL["ref"], alpha=0.35, edgecolor="none")
        # Activa: color de serie; borde de alerta solo si no cumple
        ax.barh(i, (pd.Timestamp(panel_fin) - primer).days, left=primer,
                height=0.55, color=COL["primary"], alpha=0.85,
                edgecolor=(COL["alert"] if not cumple else "none"),
                linewidth=(2 if not cumple else 0))
        ax.plot(primer, i, marker='o', color=COL["ref"], markersize=5, zorder=3)
    labels = [f"{r['Capacidad']}  ({r['Tipo'][0]})" for r in caps_a_graficar]
    ax.set_yticks(range(len(caps_a_graficar))); ax.set_yticklabels(labels); ax.invert_yaxis()
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax.axvline(pd.Timestamp(panel_inicio), color=COL["ref"], linestyle=':', linewidth=1)
    ax.axvline(pd.Timestamp(panel_fin), color=COL["ref"], linestyle=':', linewidth=1)
    legend_elements = [
        Patch(facecolor=COL["ref"], alpha=0.35, label='Pre (sin actividad)'),
        Patch(facecolor=COL["primary"], alpha=0.85, label='Activa'),
        Patch(facecolor=COL["primary"], alpha=0.85, edgecolor=COL["alert"], linewidth=2, label='Activa, no cumple C6'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', frameon=False)
    fig.suptitle("C6. Cobertura temporal por capacidad.")
    ax.set_xlabel("Periodo")
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

# C8. Co-ocurrencia entre capacidades

C8.1 masa de coactivación · C8.2 anidamiento · C8.3 VIF expandido. Las combinaciones que superan las tres verificaciones son las candidatas a términos del modelo.

In [ ]:
for cap in CAPACIDADES_MODELO:
    col_activo = f'{cap}_activo'
    if col_activo not in panel_modelo.columns:
        if cap in CAPACIDADES_CONTINUAS:
            panel_modelo = panel_modelo.withColumn(col_activo, (sf.col(cap) > 0).cast('integer'))
        else:
            panel_modelo = panel_modelo.withColumn(col_activo, sf.col(cap).cast('integer'))

n_total_c8 = panel_modelo.count()
print(f"Total PDV-meses: {n_total_c8:,}")
print(f"Capacidades en C8 ({len(CAPACIDADES_MODELO)}): {CAPACIDADES_MODELO}")
print(f"Combinaciones posibles (de 1 a {len(CAPACIDADES_MODELO)} capacidades): {2**len(CAPACIDADES_MODELO) - 1}")
print(f"\nUmbrales aplicables a C8:")
for k in ['masa_pura_min_pct', 'vif_max', 'r2_max', 'cond_xtx_max']:
    print(f"  {k}: {UMBRALES[k]}")

In [ ]:
# C8.1 Co-ocurrencia y colinealidad within entre capacidades.
# MODO_COOC: "pdv_mes" mide solapamiento temporal (meses con ambas activas; lo que ve el TWFE).
#            "pdv_max" mide presencia historica (tuvo ambas alguna vez; infla por adopcion escalonada).
MODO_COOC = "pdv_mes"
UMBRAL_COOCURRENCIA = 80
UMBRAL_COLINEAL = 0.80

for c in CAPACIDADES_MODELO:
    col = f'{c}_activo'
    if col not in panel_modelo.columns:
        if c in CAPACIDADES_CONTINUAS:
            panel_modelo = panel_modelo.withColumn(col, (sf.col(c) > 0).cast('int'))
        else:
            panel_modelo = panel_modelo.withColumn(col, sf.col(c).cast('int'))

n = len(CAPACIDADES_MODELO)

if MODO_COOC == "pdv_max":
    # Presencia historica: colapsa cada PDV a "tuvo la capacidad alguna vez"
    agg_pres = [sf.max(sf.col(f'{c}_activo')).alias(c) for c in CAPACIDADES_MODELO]
    M = panel_modelo.groupBy("id_cliente").agg(*agg_pres).select(*CAPACIDADES_MODELO).toPandas().values.astype(float)
    counts_ij = M.T @ M
    counts_i = M.sum(axis=0)
    cooc_matrix = np.divide(counts_ij, counts_i[:, None], out=np.zeros((n, n)), where=counts_i[:, None] > 0) * 100
    titulo_cooc = "% de PDVs con fila que tambien tienen columna (presencia historica)"
else:
    # Solapamiento temporal: de los meses con la fila activa, % que tambien tienen la columna activa
    counts_i_d = panel_modelo.agg(
        *[sf.sum(sf.col(f'{c}_activo')).alias(c) for c in CAPACIDADES_MODELO]
    ).collect()[0].asDict()
    counts_ij = np.zeros((n, n))
    for i, ci in enumerate(CAPACIDADES_MODELO):
        exprs = [sf.sum(sf.when((sf.col(f'{ci}_activo') == 1) & (sf.col(f'{cj}_activo') == 1), 1).otherwise(0)).alias(cj)
                 for cj in CAPACIDADES_MODELO]
        fila = panel_modelo.agg(*exprs).collect()[0].asDict()
        for j, cj in enumerate(CAPACIDADES_MODELO):
            counts_ij[i, j] = fila[cj]
    counts_i = np.array([counts_i_d[c] for c in CAPACIDADES_MODELO], dtype=float)
    cooc_matrix = np.array([[counts_ij[i, j] / counts_i[i] * 100 if counts_i[i] > 0 else 0.0
                             for j in range(n)] for i in range(n)])
    titulo_cooc = "% de meses con fila activa que tambien tienen columna activa (PDV-mes)"

# Residuos within: resta media PDV y media periodo, suma la global. Es la variacion que identifica el TWFE.
n_total_c8 = panel_modelo.count()
medias_glob = panel_modelo.agg(*[sf.avg(c).alias(c) for c in CAPACIDADES_MODELO]).collect()[0].asDict()
medias_pdv = panel_modelo.groupBy("id_cliente").agg(*[sf.avg(c).alias(f"{c}_pdv") for c in CAPACIDADES_MODELO])
medias_per = panel_modelo.groupBy("period_id").agg(*[sf.avg(c).alias(f"{c}_per") for c in CAPACIDADES_MODELO])
df_w = (panel_modelo.select(["id_cliente", "period_id"] + CAPACIDADES_MODELO)
        .join(medias_pdv, "id_cliente", "left")
        .join(medias_per, "period_id", "left"))
for c in CAPACIDADES_MODELO:
    df_w = df_w.withColumn(f"r_{c}", sf.col(c) - sf.col(f"{c}_pdv") - sf.col(f"{c}_per") + sf.lit(medias_glob[c]))

sample_frac = min(1.0, 500000 / n_total_c8)
W = df_w.sample(fraction=sample_frac, seed=42).select(*[f"r_{c}" for c in CAPACIDADES_MODELO]).toPandas()
# Kendall sobre residuos within: robusto a no normalidad y a la masa de ceros de las capacidades
corr_within = W.corr(method="kendall").values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f"C8. Co-ocurrencia ({MODO_COOC}) y colinealidad within (Kendall) entre capacidades.")

sns.heatmap(cooc_matrix, annot=True, fmt='.1f', cmap=CMAP_SEC,
            xticklabels=CAPACIDADES_MODELO, yticklabels=CAPACIDADES_MODELO,
            ax=axes[0], vmin=0, vmax=100, linewidths=0.5, linecolor='white')
axes[0].set_title(f"{titulo_cooc}\nBorde = co-ocurrencia >= {UMBRAL_COOCURRENCIA}%")
for i in range(n):
    for j in range(n):
        if i != j and cooc_matrix[i, j] >= UMBRAL_COOCURRENCIA:
            axes[0].add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor=COL["alert"], lw=3))

sns.heatmap(corr_within, annot=True, fmt='.2f', cmap=CMAP_DIV,
            xticklabels=CAPACIDADES_MODELO, yticklabels=CAPACIDADES_MODELO,
            ax=axes[1], vmin=-1, vmax=1, linewidths=0.5, linecolor='white')
axes[1].set_title(f"Correlacion Kendall sobre residuos within (PDV + periodo)\nBorde = |corr| >= {UMBRAL_COLINEAL}")
for i in range(n):
    for j in range(n):
        if i != j and abs(corr_within[i, j]) >= UMBRAL_COLINEAL:
            axes[1].add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor=COL["alert"], lw=3))

plt.tight_layout()
plt.show()

# Regla de colinealidad: del par con |corr within| >= umbral se conserva la de mayor prioridad y se excluye la otra.
excluidas_colinealidad = {}
for i in range(n):
    for j in range(i + 1, n):
        r = corr_within[i, j]
        if not np.isnan(r) and abs(r) >= UMBRAL_COLINEAL:
            excluidas_colinealidad[CAPACIDADES_MODELO[j]] = (CAPACIDADES_MODELO[i], r)

idx_keep = [i for i, c in enumerate(CAPACIDADES_MODELO) if c not in excluidas_colinealidad]
cooc_matrix = cooc_matrix[np.ix_(idx_keep, idx_keep)]
corr_within = corr_within[np.ix_(idx_keep, idx_keep)]
CAPACIDADES_MODELO = [CAPACIDADES_MODELO[i] for i in idx_keep]

print(f"Modo co-ocurrencia: {MODO_COOC}")
print(f"\nCo-ocurrencia >= {UMBRAL_COOCURRENCIA}%:")
for i, ci in enumerate(CAPACIDADES_MODELO):
    for j, cj in enumerate(CAPACIDADES_MODELO):
        if i != j and cooc_matrix[i, j] >= UMBRAL_COOCURRENCIA:
            print(f"  {ci} -> {cj}: {cooc_matrix[i, j]:.1f}%")

print(f"\nColinealidad within Kendall (|corr| >= {UMBRAL_COLINEAL}):")
if excluidas_colinealidad:
    for cap, (con, r) in excluidas_colinealidad.items():
        print(f"  Excluida {cap}: corr {r:.3f} con {con} (se conserva {con} por prioridad)")
else:
    print("  Ningun par colineal")
print(f"CAPACIDADES_MODELO tras colinealidad ({len(CAPACIDADES_MODELO)}): {CAPACIDADES_MODELO}")

In [ ]:
# Restricción estructural por dependencia
UMBRAL_DEP = 80.0

idx = {cap: k for k, cap in enumerate(CAPACIDADES_MODELO)}

deps = {}
for X in CAPACIDADES_MODELO:
    i = idx[X]
    req = set()
    for Y in CAPACIDADES_MODELO:
        if X == Y:
            continue
        j = idx[Y]
        # X anida en Y: X->Y cruza el umbral y no es mutuo (Y->X queda por debajo)
        if cooc_matrix[i, j] >= UMBRAL_DEP and cooc_matrix[j, i] < UMBRAL_DEP:
            req.add(Y)
    deps[X] = req

print(f"Dependencias por anidamiento (umbral {UMBRAL_DEP:.0f}%)")
for X in CAPACIDADES_MODELO:
    print(f"  {X:>16} -> {sorted(deps[X]) if deps[X] else 'libre'}")

def combo_valido_c8(combo):
    s = set(combo)
    return all(deps[c].issubset(s) for c in combo)

combos_posibles = [c for r in range(1, len(CAPACIDADES_MODELO) + 1)
                   for c in combinations(CAPACIDADES_MODELO, r)]
n_validas = sum(combo_valido_c8(c) for c in combos_posibles)
print(f"\nCombinaciones validas: {n_validas} de {len(combos_posibles)}")

In [ ]:
print("Combinaciones validas:")
for c in combos_posibles:
    if combo_valido_c8(c):
        print(" ", " x ".join(c))

In [ ]:
# C8.2 Patrones de tratamiento: masa de PDV-meses por patron de activacion
cols_activacion = [f'{c}_activo' for c in CAPACIDADES_MODELO]

patrones_c8 = (panel_modelo
    .groupBy(*cols_activacion)
    .count()
    .orderBy(sf.desc('count'))
    .toPandas())
patrones_c8['pct_panel'] = patrones_c8['count'] / n_total_c8 * 100
patrones_c8['n_capacidades'] = patrones_c8[cols_activacion].sum(axis=1)
patrones_c8['patron'] = patrones_c8.apply(
    lambda r: ', '.join(c for c in CAPACIDADES_MODELO if r[f'{c}_activo'] == 1) or '(ninguna)', axis=1)

n_posibles = 2**len(CAPACIDADES_MODELO)
print(f"Patrones presentes: {len(patrones_c8)} de {n_posibles}")
print(f"Patrones ausentes: {n_posibles - len(patrones_c8)}")
print(f"\nMasa por patron:")
print(f"  >= 1.0% del panel: {(patrones_c8['pct_panel'] >= 1.0).sum()}")
print(f"  0.1% a 1.0%:       {((patrones_c8['pct_panel'] >= 0.1) & (patrones_c8['pct_panel'] < 1.0)).sum()}")
print(f"  < 0.1%:            {(patrones_c8['pct_panel'] < 0.1).sum()}")
print(f"\nTop 15 por masa:")
print(patrones_c8.head(15)[['patron', 'n_capacidades', 'count', 'pct_panel']].round(3).to_string(index=False))

In [ ]:
# C8.2 Masa de co-activacion por combinacion valida tras anidamiento
if 'combo_valido_c8' not in globals():
    raise NameError("Falta la celda de anidamiento (define combo_valido_c8) antes de esta.")

combos_full = [list(c) for r in range(1, len(CAPACIDADES_MODELO) + 1)
               for c in combinations(CAPACIDADES_MODELO, r)]
combos_validos = [c for c in combos_full if combo_valido_c8(c)]

print(f"Combinaciones posibles: {len(combos_full)}")
print(f"Validas tras anidamiento: {len(combos_validos)}")
print(f"Descartadas por dependencia: {len(combos_full) - len(combos_validos)}")

# Co-activacion: PDV-meses con todas las del combo activas, las demas libres.
# Es la masa que identifica el termino de interaccion, no el patron exclusivo.
patrones_mat = patrones_c8[[f'{c}_activo' for c in CAPACIDADES_MODELO]].values
counts_patr = patrones_c8['count'].values

resultados = []
for combo in combos_validos:
    idx_combo = [CAPACIDADES_MODELO.index(c) for c in combo]
    mask = (patrones_mat[:, idx_combo] == 1).all(axis=1)
    masa_n = int(counts_patr[mask].sum())
    pct = masa_n / n_total_c8 * 100
    resultados.append({
        'combinacion': ' x '.join(combo),
        'capacidades': combo,
        'orden': len(combo),
        'masa_coact_n': masa_n,
        'pct_panel': pct,
        'pasa': pct >= UMBRALES['masa_pura_min_pct']
    })

df_masa = pd.DataFrame(resultados)
combos_masa = df_masa[df_masa['pasa']]['capacidades'].tolist()

print(f"\nCon masa de co-activacion >= {UMBRALES['masa_pura_min_pct']}%: {len(combos_masa)}")
print(f"Descartadas por masa: {len(combos_validos) - len(combos_masa)}")
print(f"\nValidas por orden:")
print(df_masa[df_masa['pasa']].groupby('orden').size().to_frame('cantidad').T.to_string())
print(f"\nDescartadas por masa (menor co-activacion):")
print(df_masa[~df_masa['pasa']].nsmallest(10, 'masa_coact_n')[['combinacion','orden','masa_coact_n','pct_panel']].round(3).to_string(index=False))

In [ ]:
# C8.3 VIF expandido: identificabilidad multivariada de las combinaciones con masa
pdvs_muestra = (panel_modelo
    .select('id_cliente').distinct()
    .orderBy(sf.rand(seed=42))
    .limit(N_PDVS_MUESTRA_VIF)
    .toPandas()['id_cliente'].tolist())

df_pd = (panel_modelo
    .filter(sf.col('id_cliente').isin(pdvs_muestra))
    .select(['id_cliente', 'period_id'] + CAPACIDADES_MODELO)
    .toPandas())
print(f"Muestra: {len(df_pd):,} filas | {df_pd['id_cliente'].nunique():,} PDVs")

pdv = df_pd['id_cliente'].values
tt = df_pd['period_id'].values

# Centrar within cada factor antes de multiplicar: evita que la interaccion correlacione de forma artificial con sus propios marginales e infle el VIF.
fac_w = {}
for c in CAPACIDADES_MODELO:
    s = df_pd[c].astype('float64')
    fac_w[c] = (s - s.groupby(pdv).transform('mean') - s.groupby(tt).transform('mean') + s.mean()).values

columnas_X = {' x '.join(combo): np.prod([fac_w[c] for c in combo], axis=0) for combo in combos_masa}
X = pd.DataFrame(columnas_X, index=df_pd.index)
print(f"Matriz X: {X.shape}")

# Demean within PDV+periodo de los productos (Gauss-Seidel)
X_d = X.astype('float64').copy()
for it in range(100):
    X_d -= X_d.groupby(pdv).transform('mean')
    m_t = X_d.groupby(tt).transform('mean')
    X_d -= m_t
    if m_t.abs().max().max() < 1e-7:
        break
print(f"Demean convergio en {it+1} iteraciones")

# VIF y R2 via inversa de X'X: VIF_j = diag(X'X) * diag((X'X)^-1)
Xv = X_d.values
XtX = Xv.T @ Xv
cond_XtX = np.linalg.cond(XtX)
if cond_XtX > UMBRALES['cond_xtx_max']:
    XtX_inv = np.linalg.pinv(XtX, rcond=1e-10)
    singular = True
else:
    XtX_inv = np.linalg.inv(XtX)
    singular = False

vif = np.diag(XtX) * np.diag(XtX_inv)
vif = np.where(vif <= 0, np.inf, vif)
r2 = np.where(np.isfinite(vif), np.clip(1 - 1/vif, 0, 1), 1.0)

sv = np.linalg.svd(Xv, compute_uv=False)
cond_num = (sv[0] / sv[-1])**2 if sv[-1] > 0 else np.inf

vif_df = pd.DataFrame({
    'termino': X_d.columns.tolist(),
    'VIF': vif,
    'R2_aux': r2
}).sort_values('VIF', ascending=False)

print(f"\nX'X {'singular (pinv)' if singular else 'no singular'} | cond(X'X) = {cond_XtX:.2e}")
print(f"Numero de condicion: {cond_num:,.0f}")
print(f"Terminos con VIF > 5: {(vif_df['VIF'] > 5).sum()}")
print(f"Terminos con VIF > {UMBRALES['vif_max']}: {(vif_df['VIF'] > UMBRALES['vif_max']).sum()}")
print(f"Terminos con R2 > {UMBRALES['r2_max']}: {(vif_df['R2_aux'] > UMBRALES['r2_max']).sum()}")
print(f"\nVIF por termino:")
print(vif_df.round(3).to_string(index=False))

In [ ]:
Xz = X_d.values
Xz = (Xz - Xz.mean(axis=0)) / (Xz.std(axis=0) + 1e-12)
cond_estandarizado = np.linalg.cond(Xz)
print(f"Numero de condicion estandarizado (sobre X, no X'X): {cond_estandarizado:,.1f}")

In [ ]:
# Filtrado por VIF respetando jerarquia y construccion de LISTA_C8
nombres_a_combos = {' x '.join(c): c for c in combos_masa}
X_actual = X_d.copy()
trazas = []

for it in range(200):
    Xv = X_actual.values
    XtX = Xv.T @ Xv
    cond_XtX = np.linalg.cond(XtX)
    XtX_inv = (np.linalg.pinv(XtX, rcond=1e-10)
               if cond_XtX > UMBRALES['cond_xtx_max'] else np.linalg.inv(XtX))
    vif_actual = np.diag(XtX) * np.diag(XtX_inv)
    vif_actual = np.where(vif_actual <= 0, np.inf, vif_actual)

    cols = list(X_actual.columns)
    altos = [(cols[k], vif_actual[k]) for k in range(len(cols)) if vif_actual[k] > UMBRALES['vif_max']]
    if not altos:
        print(f"Sin terminos con VIF > {UMBRALES['vif_max']}: nada que filtrar")
        break

    # Solo eliminable el termino que no es subconjunto de otro presente: sacar un
    # marginal de uno de orden mayor romperia la jerarquia del modelo.
    presentes = {nom: set(nombres_a_combos[nom]) for nom in cols}
    eliminables = [(nom, v) for nom, v in altos
                   if not any(presentes[nom] < presentes[otro] for otro in cols if otro != nom)]
    if not eliminables:
        print("VIF altos restantes son marginales de terminos presentes: se conservan por jerarquia")
        break

    nom_peor, vif_peor = max(eliminables, key=lambda t: t[1])
    trazas.append({'iter': it+1, 'eliminado': nom_peor, 'VIF': round(vif_peor, 2), 'restantes': len(cols)-1})
    X_actual = X_actual.drop(columns=[nom_peor])
    print(f"Iter {it+1}: saca '{nom_peor}' (VIF={vif_peor:.1f}), quedan {X_actual.shape[1]}")

# Diagnostico final de la lista
Xv = X_actual.values
XtX = Xv.T @ Xv
cond_f = np.linalg.cond(XtX)
XtX_inv = np.linalg.pinv(XtX, rcond=1e-10) if cond_f > UMBRALES['cond_xtx_max'] else np.linalg.inv(XtX)
vif_f = np.diag(XtX) * np.diag(XtX_inv)
r2_f = np.clip(1 - 1/vif_f, 0, 1)
sv = np.linalg.svd(Xv, compute_uv=False)
cond_final = (sv[0] / sv[-1])**2 if sv[-1] > 0 else np.inf

df_vif_final = pd.DataFrame({
    'termino': X_actual.columns.tolist(),
    'VIF': vif_f,
    'R2_aux': r2_f
}).sort_values('VIF', ascending=False)

LISTA_C8 = [nombres_a_combos[nom] for nom in X_actual.columns.tolist()]

print(f"\nCascada de C8:")
print(f"  Validas tras anidamiento: {len(combos_validos)}")
print(f"  Con masa de co-activacion: {len(combos_masa)}")
print(f"  En LISTA_C8 tras VIF: {len(LISTA_C8)}")
print(f"\nDiagnostico final:")
print(f"  max VIF: {df_vif_final['VIF'].max():.2f} (umbral {UMBRALES['vif_max']})")
print(f"  max R2: {df_vif_final['R2_aux'].max():.3f} (umbral {UMBRALES['r2_max']})")
print(f"  Numero de condicion: {cond_final:,.0f}")
print(f"\nLISTA_C8 ({len(LISTA_C8)} combinaciones):")
for i, combo in enumerate(LISTA_C8, 1):
    print(f"  {i:2d}. {' x '.join(combo)}")
print(f"\nVIF final por termino:")
print(df_vif_final.round(3).to_string(index=False))
if trazas:
    print(f"\nEliminaciones por VIF ({len(trazas)}):")
    print(pd.DataFrame(trazas).to_string(index=False))

# C9. Umbral mínimo de PDVs

C9.1 umbral mínimo (poder estadístico + negocio) · C9.2 umbral sostenido en el tiempo · C9.3 medibilidad por cohorte de adopción.

In [ ]:
# C9.1 Umbrales y PDVs disponibles por combinación
ALPHA = 0.05
POWER = 0.80
EFFECT_SIZE = 0.10

z_alpha = scistats.norm.ppf(1 - ALPHA / 2)
z_beta  = scistats.norm.ppf(POWER)
umbral_estadistico = int(2 * ((z_alpha + z_beta) / EFFECT_SIZE) ** 2)
total_pdvs_c9  = panel_modelo.select("id_cliente").distinct().count()
umbral_negocio = int(total_pdvs_c9 * UMBRALES['pdvs_min_pct'] / 100)
MIN_STREAK_MINIMO = UMBRALES['streak_min']
MIN_STREAK_IDEAL  = UMBRALES['streak_ideal']

print(f"Total PDVs: {total_pdvs_c9:,}")
print(f"Umbral estadístico (poder: alpha={ALPHA}, power={POWER}, efecto={EFFECT_SIZE}): {umbral_estadistico:,}")
print(f"Umbral de negocio ({UMBRALES['pdvs_min_pct']}% de la base): {umbral_negocio:,}")
print(f"Combinaciones a evaluar: {len(LISTA_C8)}")

# PDVs con la combinación coactivada en al menos un mes
aggs = []
for i, caps_c in enumerate(LISTA_C8):
    cond = sf.col(caps_c[0]) > 0
    for c in caps_c[1:]:
        cond = cond & (sf.col(c) > 0)
    aggs.append(sf.countDistinct(sf.when(cond, sf.col("id_cliente"))).alias(f"combo_{i}"))
row_c91 = panel_modelo.agg(*aggs).collect()[0]

umbral_medible = min(umbral_estadistico, umbral_negocio)
df_c91 = pd.DataFrame([
    {"combinacion": " x ".join(c), "pdvs": int(row_c91[f"combo_{i}"] or 0),
     "medible": int(row_c91[f"combo_{i}"] or 0) >= umbral_medible}
    for i, c in enumerate(LISTA_C8)
]).sort_values("pdvs", ascending=True)

fig, ax = plt.subplots(figsize=(12, max(4, len(df_c91) * 0.42)))
fig.suptitle("C9.1 PDVs disponibles por combinación")
ax.barh(df_c91["combinacion"], df_c91["pdvs"],
        color=[COL["ok"] if m else COL["alert"] for m in df_c91["medible"]],
        alpha=0.85, edgecolor="white")
ax.axvline(umbral_estadistico, color=COL["serie"][0], linestyle="--", label=f"Estadístico ({umbral_estadistico:,})")
ax.axvline(umbral_negocio, color=COL["serie"][2], linestyle="--", label=f"Negocio ({umbral_negocio:,})")
ax.set_xlabel("PDVs con la combinación coactivada (>= 1 mes)")
ax.legend()
ax.grid(axis="x")
plt.tight_layout()
plt.show()

print(df_c91.sort_values("pdvs", ascending=False).to_string(index=False))

In [ ]:
# C9.2 Umbral sostenido: racha de meses consecutivos sobre el umbral
def max_streak_consecutivo(periodos, conteos, umbral):
    max_s = actual = 0
    prev = None
    for p, c in zip(periodos, conteos):
        if c >= umbral and (prev is None or p == prev + 1):
            actual += 1
        elif c >= umbral:
            actual = 1
        else:
            actual = 0
        max_s = max(max_s, actual)
        prev = p
    return max_s


def clasificar_estado(streak_estad, streak_neg):
    if streak_estad >= MIN_STREAK_IDEAL:  return 'Ideal (estadistico)'
    if streak_estad >= MIN_STREAK_MINIMO: return 'Minimo (estadistico)'
    if streak_neg >= MIN_STREAK_IDEAL:    return 'Ideal (solo negocio)'
    if streak_neg >= MIN_STREAK_MINIMO:   return 'Minimo (solo negocio)'
    return 'No medible'


periodos_completos = (panel_modelo
    .select("period_id").distinct().orderBy("period_id")
    .toPandas()['period_id'].tolist())

# Una sola pasada: countDistinct condicional de PDVs por combinacion por periodo
aggs = []
for i, caps in enumerate(LISTA_C8):
    cond = sf.col(caps[0]) > 0
    for c in caps[1:]:
        cond = cond & (sf.col(c) > 0)
    aggs.append(sf.countDistinct(sf.when(cond, sf.col("id_cliente"))).alias(f"combo_{i}"))

conteos_pd = (panel_modelo.groupBy("period_id").agg(*aggs).orderBy("period_id").toPandas())
conteos_pd = (pd.DataFrame({'period_id': periodos_completos})
    .merge(conteos_pd, on='period_id', how='left').fillna(0))

periodos = conteos_pd['period_id'].tolist()
resultados = []
series_tiempo = {}
for i, caps in enumerate(LISTA_C8):
    nombre = ' x '.join(caps)
    conteos = conteos_pd[f'combo_{i}'].astype(int).tolist()
    series_tiempo[nombre] = pd.DataFrame({'period_id': periodos, 'n_pdvs': conteos})
    if sum(conteos) == 0:
        resultados.append({'Combinacion': nombre, 'capacidades': caps, 'Streak (estad.)': 0,
                           'Streak (negocio)': 0, 'PDVs min': 0, 'PDVs max': 0, 'Estado': 'Sin PDVs'})
        continue
    streak_estad = max_streak_consecutivo(periodos, conteos, umbral_estadistico)
    streak_neg = max_streak_consecutivo(periodos, conteos, umbral_negocio)
    resultados.append({'Combinacion': nombre, 'capacidades': caps, 'Streak (estad.)': streak_estad,
                       'Streak (negocio)': streak_neg, 'PDVs min': min(conteos), 'PDVs max': max(conteos),
                       'Estado': clasificar_estado(streak_estad, streak_neg)})

res_df = pd.DataFrame(resultados).sort_values('Streak (estad.)', ascending=False)
print()
print(res_df.drop(columns=['capacidades']).to_string(index=False))

ESTADOS_VALIDOS = ['Ideal (estadistico)', 'Minimo (estadistico)', 'Ideal (solo negocio)', 'Minimo (solo negocio)']
LISTA_C9 = res_df[res_df['Estado'].isin(ESTADOS_VALIDOS)]['capacidades'].tolist()

print(f"\nLISTA_C9 (masa temporal suficiente):")
print(f"  LISTA_C8 input: {len(LISTA_C8)}")
print(f"  Pasan C9: {len(LISTA_C9)}")
print(f"  Eliminadas: {len(LISTA_C8) - len(LISTA_C9)}")
for i, combo in enumerate(LISTA_C9, 1):
    print(f"  {i:2d}. {' x '.join(combo)}")

# Streak por combinacion: color por estado (semaforo), lineas de streak de referencia
nivel_color = {
    'Ideal (estadistico)': COL['ok'], 'Minimo (estadistico)': COL['ok'],
    'Ideal (solo negocio)': COL['warn'], 'Minimo (solo negocio)': COL['warn'],
    'No medible': COL['alert'], 'Sin PDVs': COL['alert'],
}
colores = res_df['Estado'].map(nivel_color)

fig, ax = plt.subplots(figsize=(14, max(5, len(res_df) * 0.5)))
fig.suptitle('C9. Streak maximo de periodos consecutivos con masa.')
ax.barh(res_df['Combinacion'], res_df['Streak (estad.)'], color=colores, alpha=0.85, edgecolor='white')
ax.axvline(MIN_STREAK_IDEAL, color=COL['serie'][0], linestyle='--', label=f'Ideal ({MIN_STREAK_IDEAL})')
ax.axvline(MIN_STREAK_MINIMO, color=COL['serie'][1], linestyle='--', label=f'Minimo ({MIN_STREAK_MINIMO})')
for j, (_, row) in enumerate(res_df.iterrows()):
    ax.text(row['Streak (estad.)'] + 0.2, j,
            f"e:{row['Streak (estad.)']} n:{row['Streak (negocio)']} | {row['Estado']}", va='center')
ax.set_xlabel('Streak maximo (umbral estadistico)')
ax.legend()
ax.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Series de tiempo por combinacion: barra verde/naranja/roja segun umbral superado
alto = max(umbral_estadistico, umbral_negocio)
bajo = min(umbral_estadistico, umbral_negocio)
n = len(res_df)
fig, axes = plt.subplots(n, 1, figsize=(14, 2.2 * n), sharex=False)
if n == 1:
    axes = [axes]
fig.suptitle(f'C9. PDVs simultaneos por periodo (estadistico {umbral_estadistico:,}, negocio {umbral_negocio:,}).')
for ax, (_, row) in zip(axes, res_df.iterrows()):
    serie = series_tiempo[row['Combinacion']]
    colores_b = [COL['ok'] if v >= alto else COL['warn'] if v >= bajo else COL['alert'] for v in serie['n_pdvs']]
    ax.bar(serie['period_id'], serie['n_pdvs'], color=colores_b, alpha=0.85, edgecolor='white')
    ax.axhline(umbral_estadistico, color=COL['serie'][0], linestyle='--')
    ax.axhline(umbral_negocio, color=COL['serie'][1], linestyle='--')
    ax.set_title(f"{row['Combinacion']} | {row['Estado']}")
    ax.set_ylabel('PDVs')
    ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nRESUMEN C9:")
for estado in ['Ideal (estadistico)', 'Minimo (estadistico)', 'Ideal (solo negocio)',
               'Minimo (solo negocio)', 'No medible', 'Sin PDVs']:
    nn = (res_df['Estado'] == estado).sum()
    if nn > 0:
        print(f"  {estado}: {nn}")

In [ ]:
# C9.2 Capacidades sin término en el modelo (candidatas a monitoreo)
caps_en_c9 = {c for combo in LISTA_C9 for c in combo}
sin_termino = [c for c in CAPACIDADES_MODELO if c not in caps_en_c9]
TERMINOS_MONITOREO = sorted(set(MONITOREO_C8) | set(sin_termino))

if TERMINOS_MONITOREO:
    print(f"Bajo monitoreo (sin término estable en el modelo): {TERMINOS_MONITOREO}")
    print("Se incorporan cuando su masa o racha se consolide; queda registrado en el archivo de decisiones")
else:
    print("Todas las capacidades tienen al menos un término en el modelo")

In [ ]:
# C9.3 Medibilidad por cohorte de adopción
MIN_PRE_COHORTE, MIN_POST_COHORTE = 6, 3
p_min_c93 = panel_modelo.agg(sf.min("periodo")).collect()[0][0]
p_max_c93 = panel_modelo.agg(sf.max("periodo")).collect()[0][0]

def _meses_entre(a, b):
    return (b.year - a.year) * 12 + (b.month - a.month)

COHORTES_MEDIBLES = {}
caps_c93 = []
datos_c93 = {}
for cap in CAPACIDADES_MODELO:
    df = panel_modelo
    flag = FLAGS_CENSURA.get(cap)
    if flag and flag in df.columns:
        df = df.filter(sf.col(flag) == 0)
    coh = (df.filter(sf.col(cap) > 0)
           .groupBy("id_cliente").agg(sf.min("periodo").alias("fa"))
           .groupBy("fa").count().orderBy("fa").toPandas())
    if len(coh) == 0:
        continue
    coh["pre"]  = coh["fa"].apply(lambda f: _meses_entre(p_min_c93, f))
    coh["post"] = coh["fa"].apply(lambda f: _meses_entre(f, p_max_c93))
    coh["tam_ok"]     = coh["count"] >= umbral_medible
    coh["ventana_ok"] = (coh["pre"] >= MIN_PRE_COHORTE) & (coh["post"] >= MIN_POST_COHORTE)
    coh["medible"]    = coh["tam_ok"] & coh["ventana_ok"]
    coh["causa"] = np.where(coh["medible"], "",
                   np.where(~coh["tam_ok"] & ~coh["ventana_ok"], "sin tamaño y sin ventana",
                   np.where(~coh["tam_ok"], "sin tamaño", "sin ventana")))
    caps_c93.append(cap)
    datos_c93[cap] = coh
    COHORTES_MEDIBLES[cap] = {
        "n_cohortes": int(len(coh)),
        "medibles": int(coh["medible"].sum()),
        "pct_pdvs_en_medibles": round(float(coh.loc[coh["medible"], "count"].sum() / coh["count"].sum() * 100), 1),
    }

print(f"C9.3 — Medibilidad por cohorte (tamaño >= {umbral_medible:,}, ventana >= {MIN_PRE_COHORTE} pre / {MIN_POST_COHORTE} post)")
print(pd.DataFrame([{"capacidad": c, **v} for c, v in COHORTES_MEDIBLES.items()]).to_string(index=False))

n_c93 = len(caps_c93)
if n_c93:
    ncols = min(3, n_c93)
    nrows = int(np.ceil(n_c93 / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 3.6 * nrows), squeeze=False)
    fig.suptitle("C9.3 Tamaño de cohortes de adopción (verde = medible)")
    for k, cap in enumerate(caps_c93):
        ax = axes[k // ncols][k % ncols]
        coh = datos_c93[cap]
        ax.bar(coh["fa"], coh["count"], width=20,
               color=[COL["ok"] if m else COL["alert"] for m in coh["medible"]],
               alpha=0.85, edgecolor="white")
        ax.axhline(umbral_medible, color=COL["ref"], linestyle="--", linewidth=1)
        ax.set_title(cap)
        ax.grid(axis="y")
    for k in range(n_c93, nrows * ncols):
        axes[k // ncols][k % ncols].set_visible(False)
    plt.tight_layout()
    plt.show()

# C10. Penetración alta y concentración en el valor máximo

In [ ]:
# C10. Alta penetracion y concentracion en el valor maximo
VENTANA_3 = 3
VENTANA_6 = 6

print(f"Umbral penetracion: {UMBRALES['penetracion_max']*100:.0f}% de PDVs sostenida")
print(f"Umbral saturacion: {UMBRALES['saturacion_max']*100:.0f}% de PDVs en el valor maximo (continuas)")

capacidades_c10 = sorted({c for combo in LISTA_C9 for c in combo})
print(f"Capacidades a evaluar: {capacidades_c10}")

periodos_desc = (panel_modelo.select("period_id").distinct()
    .orderBy(sf.col("period_id").desc()).limit(VENTANA_6)
    .toPandas()['period_id'].tolist())
ultimos_3 = sorted(periodos_desc[:VENTANA_3])
ultimos_6 = sorted(periodos_desc[:VENTANA_6])
print(f"Ultimos 3 meses: {ultimos_3[0]} a {ultimos_3[-1]}")
print(f"Ultimos 6 meses: {ultimos_6[0]} a {ultimos_6[-1]}")

# Penetracion de todas las capacidades en una sola pasada
aggs = [sf.countDistinct("id_cliente").alias("pdvs_totales")]
for cap in capacidades_c10:
    aggs.append(sf.countDistinct(sf.when(sf.col(cap) > 0, sf.col("id_cliente"))).alias(f"act_{cap}"))
pen = panel_modelo.groupBy("period_id").agg(*aggs).orderBy("period_id").toPandas()
for cap in capacidades_c10:
    pen[f"pct_{cap}"] = pen[f"act_{cap}"] / pen["pdvs_totales"] * 100

# Concentracion en el valor maximo (continuas): P99 como techo robusto, corte en 0.9*P99
techo = {}
muestras_hist = {}
for cap in capacidades_c10:
    if cap not in CAPACIDADES_CONTINUAS:
        continue
    activos = panel_modelo.filter(sf.col(cap) > 0)
    n_act = activos.count()
    p99 = activos.approxQuantile(cap, [0.99], 0.001)[0]
    corte = 0.9 * p99
    n_techo = activos.filter(sf.col(cap) >= corte).count()
    techo[cap] = {'p99': p99, 'corte': corte, 'pct': n_techo / n_act * 100 if n_act > 0 else 0.0}
    frac = min(1.0, 100000 / n_act) if n_act > 0 else 1.0
    muestras_hist[cap] = activos.select(cap).sample(False, frac, seed=42).toPandas()[cap].values

resultados = []
series_pen = {}
for cap in capacidades_c10:
    serie = pen[['period_id', f'pct_{cap}']].rename(columns={f'pct_{cap}': 'pct'})
    series_pen[cap] = serie
    pct_max = serie['pct'].max()
    periodo_max = serie.loc[serie['pct'].idxmax(), 'period_id']
    pct_3m = serie[serie['period_id'].isin(ultimos_3)]['pct'].mean()
    pct_6m = serie[serie['period_id'].isin(ultimos_6)]['pct'].mean()
    alta = (pct_3m >= UMBRALES['penetracion_max']*100 or pct_6m >= UMBRALES['penetracion_max']*100
            or pct_max >= UMBRALES['penetracion_max']*100)

    es_continua = cap in CAPACIDADES_CONTINUAS
    pct_techo = techo[cap]['pct'] if es_continua else None
    efecto_techo = es_continua and pct_techo >= UMBRALES['saturacion_max']*100

    if es_continua:
        if alta and efecto_techo:
            estado, nivel = 'Excluir como tratamiento, usar como covariable', 'critico'
        elif alta:
            estado, nivel = 'Alta penetracion sin concentracion en maximo', 'medio'
        elif efecto_techo:
            estado, nivel = 'Concentracion en maximo pero penetracion baja', 'medio'
        else:
            estado, nivel = 'Sin alerta', 'ok'
    else:
        estado, nivel = ('Excluir como tratamiento, alta penetracion', 'critico') if alta else ('Sin alerta', 'ok')

    resultados.append({'Capacidad': cap, 'Tipo': 'Continua' if es_continua else 'Binaria',
        'pen 3m': round(pct_3m, 1), 'pen 6m': round(pct_6m, 1), 'pen max': round(pct_max, 1),
        'periodo max': str(periodo_max),
        'pct en maximo': round(pct_techo, 1) if pct_techo is not None else None,
        'Estado': estado, 'Nivel': nivel})

res_c10 = pd.DataFrame(resultados)
print("\n" + res_c10.to_string(index=False))

CAPACIDADES_CONTROL = res_c10[res_c10['Nivel'] == 'critico']['Capacidad'].tolist()
CAPACIDADES_TRATAMIENTO = [c for c in capacidades_c10 if c not in CAPACIDADES_CONTROL]
print(f"\nCAPACIDADES_TRATAMIENTO ({len(CAPACIDADES_TRATAMIENTO)}): {CAPACIDADES_TRATAMIENTO}")
print(f"CAPACIDADES_CONTROL ({len(CAPACIDADES_CONTROL)}): {CAPACIDADES_CONTROL}")

# Penetracion por periodo
fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle('C10. Penetracion por periodo (porcentaje de PDVs con la capacidad activa).')
for i, cap in enumerate(capacidades_c10):
    serie = series_pen[cap]
    ax.plot(serie['period_id'], serie['pct'], marker='o', markersize=3,
            color=COL['serie'][i % len(COL['serie'])], label=cap)
ax.axhline(UMBRALES['penetracion_max']*100, color=COL['alert'], linestyle='--',
           label=f"Umbral ({UMBRALES['penetracion_max']*100:.0f}%)")
ax.axvspan(ultimos_6[0], ultimos_6[-1], alpha=0.1, color=COL['serie'][0])
ax.set_xlabel('Periodo')
ax.set_ylabel('Porcentaje de PDVs activos')
ax.set_ylim(0, 105)
ax.legend(loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Histograma por continua con el corte de techo: evidencia que pide el Marco
continuas_c10 = [c for c in capacidades_c10 if c in CAPACIDADES_CONTINUAS]
n = len(continuas_c10)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
if n == 1:
    axes = [axes]
fig.suptitle('C10. Distribucion de cada capacidad continua y concentracion en el valor maximo.')
for ax, cap in zip(axes, continuas_c10):
    nivel = res_c10[res_c10['Capacidad'] == cap]['Nivel'].iloc[0]
    col_barra = {'critico': COL['alert'], 'medio': COL['warn'], 'ok': COL['ok']}[nivel]
    pct = res_c10[res_c10['Capacidad'] == cap]['pct en maximo'].iloc[0]
    vals = muestras_hist[cap]
    lim = techo[cap]['p99'] * 1.1
    vals_vista = vals[vals <= lim]  # recorte solo para la vista, el porcentaje usa todos los datos
    ax.hist(vals_vista, bins=40, color=col_barra, alpha=0.85, edgecolor='white')
    ax.axvline(techo[cap]['corte'], color=COL['serie'][0], linestyle='--',
               label=f"Techo 0.9*P99 ({techo[cap]['corte']:.2f})")
    ax.set_title(f"{cap} | {pct:.1f}% en maximo")
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frecuencia')
    ax.legend()
plt.tight_layout()
plt.show()

print(f"\nRESUMEN C10:")
print(f"  A control (critico): {(res_c10['Nivel'] == 'critico').sum()}")
print(f"  Alerta (medio): {(res_c10['Nivel'] == 'medio').sum()}")
print(f"  Sin alerta (ok): {(res_c10['Nivel'] == 'ok').sum()}")

# Guardar panel para el Modelo

Escribe `PanelDiagnostico`, `PanelModelo` y el archivo de decisiones que consumen los notebooks de modelado.

In [ ]:
DECISIONES = {
    'bu': f'ARCA_{BU}',
    'universo': 'Canal Tradicional',
    'convencion_proporciones': '0-100',
    'formas': FORMA,
    'rezagos': REZAGOS,
    'combinaciones_estructura': [list(c) for c in LISTA_C9],
    'capacidades_tratamiento': CAPACIDADES_TRATAMIENTO,
    'capacidades_control': CAPACIDADES_CONTROL,
    'terminos_monitoreo': TERMINOS_MONITOREO,
    'flags_censura': {k: v for k, v in FLAGS_CENSURA.items() if v in panel_modelo.columns},
    'umbrales_aplicados': UMBRALES,
    'fecha_generacion': pd.Timestamp.now().isoformat(),
    'diagnosticos': {
        'n_pdvs_universo': total_pdvs_c9,
        'combos_masa_c81': len(combos_masa),
        'combos_tras_anidamiento_c82': len(combos_c82),
        'en_lista_c8_vif': len(LISTA_C8),
        'pasan_streak_c9': len(LISTA_C9),
        'max_vif_final': float(df_vif_final['VIF'].max()),
        'numero_condicion_final': float(cond_final),
        'anidados': anidados,
        'e4': DIAGNOSTICO_E4,
        'cohortes_medibles': COHORTES_MEDIBLES,
    },
}

output_path = f"abfss://{containerName}@{storageAccountName}.dfs.core.windows.net/CTG/{BU}/Modelo/capacidades"

(spark.createDataFrame([(json.dumps(DECISIONES, indent=2,
                                    default=lambda o: o.item() if hasattr(o, 'item') else str(o)),)], ['contenido'])
      .coalesce(1)
      .write.mode('overwrite')
      .text(output_path))

print(f"Decisiones persistidas en: {output_path}")
for k in ['formas', 'rezagos', 'combinaciones_estructura', 'capacidades_tratamiento',
          'capacidades_control', 'terminos_monitoreo']:
    print(f"  {k}: {DECISIONES[k]}")

In [ ]:
base_path = f"abfss://{containerName}@{storageAccountName}.dfs.core.windows.net/CTG/{BU}"

if USE_MUESTRA:
    n_pdvs_muestra = panel_modelo.select("id_cliente").distinct().count()
    sufijo = f"_muestra_{n_pdvs_muestra}"
    print(f"Modo muestra: guardando con sufijo '{sufijo}'")
else:
    sufijo = ""

path_diag = f"{base_path}/PanelDiagnostico{sufijo}/parquet/"
panel_diagnostico.write.mode("overwrite").parquet(path_diag)
print(f"PanelDiagnostico escrito | {path_diag}")

path_mod = f"{base_path}/PanelModelo{sufijo}/parquet/"
panel_modelo.write.mode("overwrite").parquet(path_mod)
print(f"PanelModelo escrito | {path_mod} | filas: {panel_modelo.count():,} | PDVs: {panel_modelo.select('id_cliente').distinct().count():,}")

In [ ]:
spark.catalog.clearCache()